<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="/ipynb/zh-CN/NLP/06-evaluation.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to NLP guideline](Natural-Language-Processing.html)


## **Evaluation in NLP** {#evaluation-in-nlp}

Evaluation in NLP means measuring whether a model actually solves the language task we care about. Training loss tells us whether the model is fitting the training objective, but evaluation asks a broader question: does the model behave correctly on unseen text, under realistic conditions, with the kind of errors that matter for the task?

A good evaluation design connects four things:

```text
task definition -> dataset split -> metric choice -> error analysis
```

For example, a sentiment classifier may be evaluated with accuracy or F1. A named entity recognition system may need entity-level span F1. A language model may use perplexity during development, but generated outputs still need human inspection. A retrieval system may care about whether the correct document appears in the top 5 results, not whether every document is classified perfectly.

The central idea is that evaluation is not just a number. It is a decision tool. It tells us whether a model is ready, where it fails, and what kind of improvement is worth making next.

### **What Does Evaluation Mean in NLP?** {#what-does-evaluation-mean-in-nlp}

Evaluation is the process of comparing model outputs with some standard of correctness. That standard may be a gold label, a reference translation, a human preference judgment, a retrieval relevance label, or a task-specific business rule.

In supervised NLP tasks, evaluation often looks simple:

```text
input text -> model prediction -> compare with gold label -> compute metric
```

For classification, the gold label may be `positive` or `negative`. For NER, the gold output may be entity spans such as `(Barack Obama, PERSON)`. For summarization, the gold output may be one or more reference summaries. For open-ended generation, there may be no single correct answer, so evaluation becomes more complicated.

This is why NLP evaluation must always be tied to the task formulation. The same model output may be good under one metric and poor under another. A generated summary may have high word overlap with the reference but still omit the key fact. A chatbot answer may sound fluent but be factually wrong. A classifier may have high accuracy while failing on a minority class.

A useful mental model is:

| Question | Evaluation Meaning |
|---|---|
| Did the model match the gold answer? | automatic correctness |
| Did the model help the user? | task usefulness |
| Did the model fail on important subgroups? | robustness and fairness |
| Did the metric hide important errors? | metric validity |
| Can the result be trusted on future data? | generalization |

In NLP, evaluation has three common levels.

| Level | What It Measures | Example |
|---|---|---|
| instance-level | one prediction at a time | this review was classified correctly |
| aggregate-level | performance over a dataset | macro F1 over all classes |
| slice-level | performance on a subset | F1 on long documents, rare entities, or non-native English text |

The last level is especially important. A single average score can hide systematic weaknesses. A model may perform well overall but fail on long sentences, rare labels, code-mixed text, negation, or domain-specific vocabulary.

<details>
<summary>Python Simple Evaluation Loop</summary>

```python
from sklearn.metrics import accuracy_score, f1_score

texts = [
    "The explanation was clear and useful.",
    "The model completely missed the point.",
    "The tutorial was okay but too short."
]

gold_labels = ["positive", "negative", "neutral"]
pred_labels = ["positive", "negative", "positive"]

accuracy = accuracy_score(gold_labels, pred_labels)
macro_f1 = f1_score(gold_labels, pred_labels, average="macro")

print("accuracy:", accuracy)
print("macro F1:", macro_f1)
```

</details>

This tiny example already shows why multiple metrics matter. Accuracy counts exact matches. Macro F1 gives each class equal importance, which can reveal weaker performance on minority labels.

### **Evaluation Pipeline** {#evaluation-pipeline}

An evaluation pipeline is the full process used to produce trustworthy performance estimates. It starts before the model is trained. If the dataset split is wrong, the labels are inconsistent, or the test set is used repeatedly for tuning, even a sophisticated metric can give a misleading result.

A practical NLP evaluation pipeline usually follows this structure:

```text
collect data
-> define labels and task format
-> split train / validation / test
-> train model on training set
-> tune decisions on validation set
-> evaluate once on test set
-> inspect errors and report limitations
```

The goal is to simulate future use. The test set should represent examples the model has not seen and should not have influenced model selection.

#### **Train / Validation / Test Split** {#train-validation-test-split}

A train / validation / test split separates the dataset into three parts with different responsibilities.

| Split | Used For | Should Influence Model Parameters? | Should Influence Model Selection? |
|---|---|---|---|
| training set | learning parameters | yes | indirectly |
| validation set | tuning hyperparameters and decisions | no direct gradient updates | yes |
| test set | final unbiased evaluation | no | no |

The training set is used to update model parameters. The validation set is used to choose learning rate, number of epochs, model size, threshold, decoding strategy, or regularization. The test set is used only at the end to estimate final generalization.

A common split might be:

```text
80% training
10% validation
10% test
```

But the right split depends on dataset size and task risk. For small datasets, cross-validation may be more reliable. For time-sensitive data, chronological splitting may be better than random splitting. For user-level data, examples from the same user should not appear in both train and test.

In NLP, splitting can be surprisingly tricky. If duplicate or near-duplicate texts appear across splits, the test score may be inflated. If one document is chunked into many passages, passages from the same document should usually stay in the same split. If the task is dialogue modeling, turns from the same conversation should not leak across train and test.

<details>
<summary>Python Stratified Train / Validation / Test Split</summary>

```python
from sklearn.model_selection import train_test_split

texts = ["text example ..."] * 1000
labels = [0, 1] * 500

# Step 1: create a held-out test set.
# stratify keeps the class distribution similar across splits.
train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts,
    labels,
    test_size=0.10,
    random_state=42,
    stratify=labels
)

# Step 2: split the remaining training data into train and validation.
train_texts, valid_texts, train_labels, valid_labels = train_test_split(
    train_texts,
    train_labels,
    test_size=0.1111,  # 0.1111 of 90% is about 10% of the full dataset
    random_state=42,
    stratify=train_labels
)

print(len(train_texts), len(valid_texts), len(test_texts))
```

</details>

The most important rule is simple: never tune on the test set. If the test set is checked repeatedly during development, it slowly becomes part of the training process, even if no gradient is computed on it.

#### **Cross-Validation** {#cross-validation}

Cross-validation is an evaluation strategy where the dataset is split into multiple folds. The model is trained and evaluated multiple times, each time using a different fold as the validation fold.

> Image for k-fold cross-validation:
>
> ![K-fold cross-validation diagram](https://upload.wikimedia.org/wikipedia/commons/b/b5/K-fold_cross_validation_EN.svg)
>
> Source: [Wikipedia - Cross-validation](https://en.wikipedia.org/wiki/Cross-validation_(statistics))

In k-fold cross-validation, the dataset is divided into $k$ parts. Each fold gets one chance to act as the validation set, while the remaining $k-1$ folds are used for training. The final score is usually the average across folds.

```text
fold 1: train on folds 2-5, validate on fold 1
fold 2: train on folds 1,3,4,5, validate on fold 2
fold 3: train on folds 1,2,4,5, validate on fold 3
...
```

The benefit is that every example is used for validation exactly once. This is helpful when the dataset is small and a single random split may be unreliable.

The cost is computation. If we use 5-fold cross-validation, we train 5 models. For large Transformer models, this can be expensive. In that case, a single validation split plus careful error analysis may be more practical.

<details>
<summary>Python Stratified K-Fold Evaluation</summary>

```python
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np

texts = np.array(["text example ..."] * 1000)
labels = np.array([0, 1] * 500)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []

for fold_id, (train_idx, valid_idx) in enumerate(skf.split(texts, labels), start=1):
    train_texts = texts[train_idx]
    valid_texts = texts[valid_idx]
    train_labels = labels[train_idx]
    valid_labels = labels[valid_idx]

    # Placeholder: train your model on train_texts / train_labels.
    # model = train_model(train_texts, train_labels)
    # predictions = model.predict(valid_texts)

    predictions = valid_labels.copy()  # placeholder for demonstration
    score = f1_score(valid_labels, predictions, average="macro")
    fold_scores.append(score)

    print(f"fold={fold_id}, macro_f1={score:.4f}")

print("mean macro F1:", np.mean(fold_scores))
print("std macro F1:", np.std(fold_scores))
```

</details>

Cross-validation is best for model comparison and small-data reliability. It is less convenient for final production evaluation, where a separate held-out test set is still useful.

#### **Development Set vs Test Set** {#development-set-vs-test-set}

The development set, often called the validation set or dev set, is used during model development. The test set is used after development decisions are finished.

The difference is not about file names. It is about decision-making.

```text
dev set:  allowed to influence choices
test set: should only measure the final chosen system
```

The dev set can guide decisions such as:

| Decision | Example |
|---|---|
| hyperparameters | learning rate, batch size, dropout |
| training duration | number of epochs, early stopping point |
| architecture choice | BiLSTM vs Transformer encoder |
| threshold choice | probability cutoff for classification |
| decoding choice | beam size, temperature, top-p |
| preprocessing | max sequence length, normalization rules |

The test set should not guide these choices. If we choose the model that performs best on the test set, the reported test score becomes optimistic. It no longer estimates performance on truly unseen future data.

A clean workflow is:

```text
try many models on dev set
choose one final model
run once on test set
report test result
```

In research papers, the test set is usually the number reported in the main results table. In real projects, the test set may be supplemented with live A/B tests, human review, or domain-specific evaluation.

#### **Data Leakage** {#data-leakage}

Data leakage happens when information that should not be available at prediction time enters the training or model-selection process. Leakage makes evaluation look better than it really is.

Common leakage patterns in NLP include:

| Leakage Type | Example |
|---|---|
| duplicate leakage | same sentence appears in train and test |
| document leakage | chunks from the same document appear in different splits |
| user leakage | same user's messages appear in train and test |
| preprocessing leakage | TF-IDF vocabulary fitted on the full dataset before splitting |
| label leakage | feature contains the answer or a proxy for the answer |
| temporal leakage | future data used to predict past events |

Leakage is dangerous because the model may appear excellent during evaluation but fail in deployment.

For example, this is wrong:

```text
fit vectorizer on all data -> split -> train -> evaluate
```

The vectorizer has already seen words from the test set. The correct order is:

```text
split -> fit vectorizer on train only -> transform validation/test
```

Using a pipeline helps prevent preprocessing leakage.

<details>
<summary>Python Avoiding Preprocessing Leakage with Pipeline</summary>

```python
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

pipeline = Pipeline([
    # TfidfVectorizer is fitted only on the training split inside pipeline.fit().
    ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1, 2))),

    # LogisticRegression receives only the train-fitted TF-IDF features.
    ("classifier", LogisticRegression(max_iter=1000))
])

pipeline.fit(train_texts, train_labels)
pred_labels = pipeline.predict(test_texts)

print(classification_report(test_labels, pred_labels))
```

</details>

For NLP datasets, it is also useful to check duplicates and near duplicates before trusting a score.

<details>
<summary>Python Simple Duplicate Leakage Check</summary>

```python
train_set = set(train_texts)
test_set = set(test_texts)

overlap = train_set.intersection(test_set)

print("number of exact duplicate texts across train/test:", len(overlap))

if overlap:
    print("Example duplicate:", next(iter(overlap)))
```

</details>

Leakage can be subtle. Exact duplicate checks are not enough when paraphrases, shared document IDs, or repeated users are involved. In serious evaluations, split by group identifiers such as `document_id`, `user_id`, or `conversation_id` when those identities matter.

#### **Human Evaluation vs Automatic Evaluation** {#human-evaluation-vs-automatic-evaluation}

Automatic evaluation uses metrics computed by code. Human evaluation uses people to judge model outputs. Both are important, but they answer different questions.

Automatic metrics are fast, cheap, reproducible, and useful during development. Human evaluation is slower and more expensive, but it can judge qualities that are difficult to capture with a simple metric, such as helpfulness, factuality, coherence, fluency, and whether an answer actually satisfies the user's intent.

| Evaluation Type | Strength | Weakness | Best For |
|---|---|---|---|
| automatic evaluation | fast, repeatable, scalable | may miss meaning or usefulness | classification, NER, retrieval, development loops |
| human evaluation | captures nuanced quality | expensive, subjective, slower | summarization, dialogue, open-ended generation |
| hybrid evaluation | combines scale and judgment | requires careful design | production NLP systems |

For classification tasks, automatic metrics are often enough if labels are reliable. For open-ended generation, automatic metrics are usually incomplete. A summary can have low word overlap but still be good. A chatbot answer can be fluent but factually wrong. A translation can be semantically correct with different wording from the reference.

Human evaluation should be designed carefully. Annotators need clear criteria, examples, and sometimes multiple independent ratings.

A simple human evaluation rubric might look like this:

| Criterion | Question | Score Range |
|---|---|---|
| relevance | does the answer address the prompt? | 1-5 |
| factuality | is the answer factually correct? | 1-5 |
| completeness | does it include the necessary information? | 1-5 |
| fluency | is the language natural and readable? | 1-5 |
| safety | does it avoid harmful or inappropriate content? | pass/fail |

<details>
<summary>Python Aggregating Human Evaluation Scores</summary>

```python
import pandas as pd

ratings = pd.DataFrame({
    "example_id": [1, 1, 2, 2, 3, 3],
    "annotator": ["A", "B", "A", "B", "A", "B"],
    "relevance": [5, 4, 3, 3, 2, 2],
    "factuality": [4, 4, 3, 2, 1, 2],
    "fluency": [5, 5, 4, 4, 3, 3]
})

summary = ratings.groupby("example_id")[["relevance", "factuality", "fluency"]].mean()
summary["overall"] = summary.mean(axis=1)

print(summary)
print("average overall score:", summary["overall"].mean())
```

</details>

Human evaluation is not automatically better than automatic evaluation. If the rubric is vague or annotators disagree heavily, human scores can be noisy. A good evaluation pipeline often uses automatic metrics for fast iteration and human evaluation for final quality checks or open-ended tasks.

### **Classification Evaluation** {#classification-evaluation}

Classification evaluation measures how well an NLP model assigns text to discrete labels. This includes sentiment classification, intent detection, topic classification, spam detection, toxicity detection, stance detection, news categorization, and many other tasks where the output is a class label.

The NLP-specific challenge is that labels often carry different practical risks. Misclassifying a neutral review as positive may be acceptable in a toy dataset, but missing a toxic comment, a self-harm message, or a high-priority customer complaint can be much more serious. This is why classification evaluation should not stop at a single accuracy number.

A typical NLP classification pipeline looks like this:

```text
text -> tokenizer / features -> classifier -> label probabilities -> predicted label -> evaluation metric
```

For example:

```text
Input:  "I waited two hours and nobody helped me."
Task:   customer support intent classification
Gold:   complaint
Pred:   general_question
Error:  the model missed an urgent negative support intent
```

The model made only one label error, but the downstream consequence matters. Good evaluation therefore asks both "how many predictions are correct?" and "which mistakes are being made?"

#### **Accuracy** {#accuracy}

Accuracy is the proportion of examples whose predicted label exactly matches the gold label.

$$
\text{Accuracy} = \frac{\text{number of correct predictions}}{\text{total number of predictions}}
$$

If a model predicts 90 out of 100 labels correctly, its accuracy is 0.90. This is easy to understand and useful when classes are balanced and all mistakes have similar cost.

In NLP, those assumptions often fail. Consider a toxicity detector where only 5% of comments are toxic. A model that always predicts `non_toxic` gets 95% accuracy, but it is useless for detecting harmful content.

```text
Dataset: 950 non-toxic comments, 50 toxic comments
Model: always predicts non-toxic
Accuracy: 950 / 1000 = 95%
Toxic recall: 0 / 50 = 0%
```

This is why accuracy is a good first metric but a dangerous final metric for imbalanced NLP tasks.

<details>
<summary>Python Accuracy on an NLP Classification Example</summary>

```python
from sklearn.metrics import accuracy_score

texts = [
    "Great explanation, very helpful!",
    "This is offensive and hateful.",
    "I need help resetting my password.",
    "The article is about climate policy."
]

gold_labels = ["positive", "toxic", "support", "topic_news"]
pred_labels = ["positive", "non_toxic", "support", "topic_news"]

accuracy = accuracy_score(gold_labels, pred_labels)

print("accuracy:", accuracy)
```

</details>

Accuracy answers: "How often is the exact label correct?" It does not answer whether rare or high-risk classes are handled well.

#### **Precision** {#precision}

Precision measures how many predicted positive cases are actually correct.

$$
\text{Precision} = \frac{TP}{TP + FP}
$$

Here, $TP$ means true positives: examples correctly predicted as the target class. $FP$ means false positives: examples incorrectly predicted as the target class.

For NLP, precision is important when false alarms are costly. In spam detection, low precision means many normal emails are incorrectly sent to spam. In toxic comment detection, low precision means safe comments may be unfairly flagged. In intent detection, low precision for `refund_request` means the system may route ordinary questions to the refund workflow.

```text
High precision for "toxic":
when the model says a comment is toxic, it is usually right.

Low precision for "toxic":
the model flags many harmless comments as toxic.
```

<details>
<summary>Python Precision for a Target NLP Class</summary>

```python
from sklearn.metrics import precision_score

# Binary toxicity example: 1 = toxic, 0 = non-toxic
gold = [0, 0, 0, 1, 1, 0, 1, 0]
pred = [0, 1, 0, 1, 0, 0, 1, 1]

precision = precision_score(gold, pred, pos_label=1)

print("toxic precision:", precision)
```

</details>

Precision is the right metric to emphasize when incorrect positive predictions are disruptive, expensive, or unfair.

#### **Recall** {#recall}

Recall measures how many actual positive cases the model successfully finds.

$$
\text{Recall} = \frac{TP}{TP + FN}
$$

Here, $FN$ means false negatives: examples that truly belong to the target class but were missed by the model.

In NLP, recall matters when missing a class is costly. For example, in safety moderation, missing toxic or self-harm content may be worse than reviewing a few extra false alarms. In customer support routing, missing urgent complaints can damage user experience. In medical text classification, missing a relevant symptom mention can be serious.

```text
High recall for "urgent_complaint":
most urgent complaints are caught.

Low recall for "urgent_complaint":
many urgent complaints are treated like normal messages.
```

<details>
<summary>Python Recall for a Target NLP Class</summary>

```python
from sklearn.metrics import recall_score

# Binary urgent complaint detection: 1 = urgent, 0 = normal
gold = [0, 1, 1, 0, 0, 1, 0, 1]
pred = [0, 1, 0, 0, 0, 1, 0, 0]

recall = recall_score(gold, pred, pos_label=1)

print("urgent complaint recall:", recall)
```

</details>

Recall is the right metric to emphasize when missed positives are more harmful than false positives.

#### **F1 Score** {#f1-score}

F1 score combines precision and recall into one number. It is the harmonic mean of precision and recall:

$$
F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
$$

The harmonic mean punishes imbalance. If precision is high but recall is very low, F1 will still be low. This makes F1 useful when we care about both false positives and false negatives.

For example, a toxicity classifier with precision 0.90 and recall 0.20 is not balanced. It is careful when it flags toxicity, but it misses most toxic examples. The F1 score makes that weakness visible.

```text
precision high, recall low -> model is conservative
precision low, recall high -> model is aggressive
both high -> model is useful
```

<details>
<summary>Python Precision, Recall, and F1 Together</summary>

```python
from sklearn.metrics import precision_recall_fscore_support

labels = ["billing", "technical", "complaint"]

gold = ["billing", "technical", "complaint", "complaint", "billing", "technical"]
pred = ["billing", "technical", "billing", "complaint", "billing", "complaint"]

precision, recall, f1, support = precision_recall_fscore_support(
    gold,
    pred,
    labels=labels,
    zero_division=0
)

for label, p, r, f, s in zip(labels, precision, recall, f1, support):
    print(f"{label:10s} precision={p:.2f} recall={r:.2f} f1={f:.2f} support={s}")
```

</details>

F1 is widely used for NLP classification because many NLP datasets are not perfectly balanced and because class-specific mistakes are often more informative than raw accuracy.

#### **Macro, Micro, and Weighted Averaging** {#macro-micro-and-weighted-averaging}

For multi-class NLP classification, precision, recall, and F1 can be computed per class. But we often need one summary score. Macro, micro, and weighted averaging summarize classes differently.

| Averaging Method | How It Works | What It Emphasizes |
|---|---|---|
| macro average | compute metric per class, then average equally | minority classes matter equally |
| micro average | count global TP/FP/FN, then compute metric | frequent classes dominate |
| weighted average | average per-class scores weighted by support | class frequency matters, but per-class view remains |

Macro F1 is especially important in NLP when rare labels matter. In intent classification, labels like `cancel_subscription` or `account_hacked` may be rare but important. Micro F1 may look good if the model performs well on common intents like `general_question`.

```text
macro F1 asks: does the model treat each class well?
micro F1 asks: does the model classify most examples well?
weighted F1 asks: how good is the model after accounting for class frequency?
```

<details>
<summary>Python Macro, Micro, and Weighted F1</summary>

```python
from sklearn.metrics import f1_score, classification_report

# Intent classification with an imbalanced label distribution.
gold = [
    "general_question", "general_question", "general_question", "general_question",
    "refund_request", "refund_request",
    "account_hacked"
]

pred = [
    "general_question", "general_question", "general_question", "refund_request",
    "general_question", "refund_request",
    "general_question"
]

print("macro F1:   ", f1_score(gold, pred, average="macro"))
print("micro F1:   ", f1_score(gold, pred, average="micro"))
print("weighted F1:", f1_score(gold, pred, average="weighted"))

print(classification_report(gold, pred, zero_division=0))
```

</details>

For NLP model reports, macro F1 plus a per-class table is often more informative than accuracy alone.

#### **Confusion Matrix** {#confusion-matrix}

A confusion matrix shows which labels are confused with which other labels. Rows usually represent gold labels, and columns represent predicted labels.

For NLP classification, the confusion matrix is often more useful than a single metric because it reveals semantic confusion. A sentiment model may confuse `neutral` with `positive`. An intent model may confuse `billing_problem` with `refund_request`. A topic classifier may confuse `politics` with `economics`.

| Gold \ Predicted | positive | neutral | negative |
|---|---:|---:|---:|
| positive | 42 | 6 | 2 |
| neutral | 10 | 25 | 8 |
| negative | 1 | 7 | 39 |

This matrix says the model is mostly correct on the diagonal, but it often confuses neutral text with positive or negative sentiment. That is a common NLP issue because neutral language can contain both positive and negative words without expressing strong sentiment.

<details>
<summary>Python Confusion Matrix for Sentiment Classification</summary>

```python
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt

labels = ["positive", "neutral", "negative"]

gold = [
    "positive", "positive", "neutral", "negative", "neutral",
    "negative", "positive", "neutral", "negative", "neutral"
]

pred = [
    "positive", "neutral", "neutral", "negative", "positive",
    "negative", "positive", "negative", "neutral", "neutral"
]

cm = confusion_matrix(gold, pred, labels=labels)

display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
display.plot(cmap="Blues", values_format="d")
plt.title("Sentiment Classification Confusion Matrix")
plt.show()
```

</details>

A confusion matrix is the bridge from metric reporting to error analysis. It tells us where to inspect examples.

#### **ROC-AUC and PR-AUC** {#roc-auc-and-pr-auc}

ROC-AUC and PR-AUC evaluate binary classifiers across many possible thresholds. This matters because many NLP classifiers produce probabilities or scores before converting them into labels.

For example, a toxicity detector may output:

```text
comment -> toxicity probability = 0.72
```

A threshold converts this probability into a label:

```text
if probability >= 0.50 -> toxic
else -> non_toxic
```

Changing the threshold changes precision and recall. A lower threshold catches more toxic comments but creates more false positives. A higher threshold reduces false positives but misses more toxic comments.

ROC curves plot true positive rate against false positive rate:

$$
\text{TPR} = \frac{TP}{TP + FN}, \qquad
\text{FPR} = \frac{FP}{FP + TN}
$$

PR curves plot precision against recall. For imbalanced NLP tasks, PR-AUC is often more informative than ROC-AUC because it focuses on performance for the positive class.

| Metric | Best Use | NLP Example |
|---|---|---|
| ROC-AUC | ranking positives above negatives when classes are not extremely imbalanced | spam detection baseline comparison |
| PR-AUC | positive class is rare and important | toxicity, abuse, urgent complaint detection |
| thresholded F1 | choosing a practical operating point | pick moderation threshold |

<details>
<summary>Python ROC-AUC and PR-AUC for Toxicity Detection</summary>

```python
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

# 1 = toxic, 0 = non-toxic
gold = [0, 0, 0, 1, 1, 0, 1, 0, 0, 1]

# Model scores before thresholding.
toxic_scores = [0.05, 0.20, 0.10, 0.82, 0.61, 0.40, 0.77, 0.12, 0.33, 0.55]

roc_auc = roc_auc_score(gold, toxic_scores)
pr_auc = average_precision_score(gold, toxic_scores)

print("ROC-AUC:", roc_auc)
print("PR-AUC:", pr_auc)

precision, recall, thresholds = precision_recall_curve(gold, toxic_scores)

for p, r, t in zip(precision[:-1], recall[:-1], thresholds):
    print(f"threshold={t:.2f}, precision={p:.2f}, recall={r:.2f}")
```

</details>

In NLP classification, threshold tuning should be guided by the task. A moderation system may prefer high recall. A user-facing auto-action system may require high precision. A triage system may choose a threshold that sends uncertain cases to human review.

A practical summary:

| Metric | What It Answers | NLP-Specific Caution |
|---|---|---|
| accuracy | how often is the label exactly right? | can hide minority-class failure |
| precision | when the model predicts a class, how often is it right? | important for costly false alarms |
| recall | how many true cases did the model find? | important for safety or urgent classes |
| F1 | are precision and recall balanced? | useful but hides threshold and class-specific details |
| macro F1 | does the model perform well across labels? | good for imbalanced intent/topic datasets |
| confusion matrix | which labels are confused? | essential for semantic error analysis |
| ROC-AUC | does the model rank positives above negatives? | can look optimistic under heavy imbalance |
| PR-AUC | how well does the model find rare positives? | better for rare toxic/spam/urgent classes |

For NLP classification, a strong evaluation report should include at least one aggregate metric, per-class metrics, a confusion matrix, and a short error analysis with real text examples.

### **Sequence Labeling Evaluation** {#sequence-labeling-evaluation}

Sequence labeling evaluation measures tasks where the model assigns a label to each token in a sequence. Unlike ordinary text classification, the output is not one label for the whole sentence. The output is a label sequence aligned with the input tokens.

Common NLP sequence labeling tasks include:

| Task | Input | Output |
|---|---|---|
| POS tagging | tokens in a sentence | noun, verb, adjective, etc. |
| Named Entity Recognition | sentence or document | PER, ORG, LOC, DATE spans |
| Slot filling | user utterance | destination, date, time, product |
| Chunking | sentence | noun phrase or verb phrase chunks |
| Biomedical entity extraction | clinical or research text | disease, gene, drug, symptom spans |

The key evaluation challenge is that token-level correctness and span-level correctness are not the same thing. A model can get most individual token tags right but still fail to extract the correct entity boundary.

For example:

```text
Text:  Barack Obama visited New York City
Gold:  [Barack Obama]PER visited [New York City]LOC
Pred:  [Barack]PER Obama visited [New York]LOC City
```

Many token labels are correct, but both extracted entities are boundary errors. For NER or slot filling, boundary errors matter because downstream systems usually consume complete spans, not isolated token tags.

> Image for BIO tagging:
>
> ![BIO tagging sequence labeling example](https://assets.mbrenndoerfer.com/notebooks/bio_tagging_files/bio-tagging-sequence-visualization.png)
>
> Source: [BIO Tagging: Encoding Entity Boundaries for Sequence Labeling](https://mbrenndoerfer.com/writing/bio-tagging-sequence-labeling-ner)

The figure shows why BIO tagging is useful: each token receives a tag, and the tag sequence can be decoded back into entity spans.

#### **Token-Level Accuracy** {#token-level-accuracy}

Token-level accuracy measures the percentage of tokens whose predicted tag exactly matches the gold tag.

$$
\text{Token Accuracy} = \frac{\text{number of correctly tagged tokens}}{\text{total number of tokens}}
$$

If a sentence has 10 tokens and the model predicts 9 token tags correctly, token-level accuracy is 90%.

This metric is easy to compute and useful for tasks like POS tagging, where each token's label is often meaningful on its own. But for NER, token accuracy can be misleading because most tokens are usually `O`, meaning outside any entity. A model that predicts `O` for everything may get high token accuracy while extracting no entities.

```text
Gold: O O O B-PER I-PER O O B-LOC O O
Pred: O O O O     O     O O O     O O

Many O tokens are correct.
But the model found zero entities.
```

So token accuracy answers: "How often is each tag correct?" It does not answer: "Did the model extract complete entities correctly?"

<details>
<summary>Python Token-Level Accuracy</summary>

```python
from sklearn.metrics import accuracy_score

# Each sentence is represented as a list of BIO tags.
gold_tags = [
    ["B-PER", "I-PER", "O", "B-LOC", "I-LOC"],
    ["O", "B-ORG", "I-ORG", "O"]
]

pred_tags = [
    ["B-PER", "I-PER", "O", "B-LOC", "O"],
    ["O", "B-ORG", "O", "O"]
]

# Flatten sentence-level tag lists into one token-level list.
gold_flat = [tag for sentence in gold_tags for tag in sentence]
pred_flat = [tag for sentence in pred_tags for tag in sentence]

token_accuracy = accuracy_score(gold_flat, pred_flat)

print("token-level accuracy:", token_accuracy)
```

</details>

Token-level accuracy is a useful sanity check, but it should not be the main metric for NER-style tasks where exact span extraction matters.

#### **Entity-Level Precision, Recall, and F1** {#entity-level-precision-recall-and-f1}

Entity-level evaluation treats each complete entity span as the prediction unit. A predicted entity is correct only if its start boundary, end boundary, and entity type all match the gold annotation.

An entity can be represented as:

```text
(start_token_index, end_token_index, entity_type)
```

where `end_token_index` is usually exclusive, following Python convention.

For example:

```text
Tokens: Barack Obama visited New York City
Gold entities:
(0, 2, PER) -> Barack Obama
(3, 6, LOC) -> New York City
```

Entity-level precision, recall, and F1 use the same general formulas as classification, but the counted items are spans instead of tokens.

$$
\text{Precision} = \frac{\text{correct predicted entities}}{\text{all predicted entities}}
$$

$$
\text{Recall} = \frac{\text{correct predicted entities}}{\text{all gold entities}}
$$

$$
F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
$$

The intuition is direct:

```text
precision: when the model extracts an entity, how often is it exactly right?
recall:    how many gold entities did the model find?
F1:        how well does it balance both?
```

<details>
<summary>Python Entity-Level Precision, Recall, and F1</summary>

```python
def bio_to_spans(tags):
    """Convert a BIO tag sequence into entity spans.

    Returns a set of tuples: (start, end, entity_type)
    where end is exclusive.
    """
    spans = set()
    start = None
    entity_type = None

    for i, tag in enumerate(tags + ["O"]):  # sentinel O closes any open entity
        if tag == "O":
            if entity_type is not None:
                spans.add((start, i, entity_type))
                start = None
                entity_type = None
            continue

        prefix, current_type = tag.split("-", 1)

        if prefix == "B":
            # Starting a new entity closes the previous one.
            if entity_type is not None:
                spans.add((start, i, entity_type))
            start = i
            entity_type = current_type

        elif prefix == "I":
            # Continue only if the entity type matches.
            # Otherwise, treat malformed I-X as a new B-X entity.
            if entity_type != current_type:
                if entity_type is not None:
                    spans.add((start, i, entity_type))
                start = i
                entity_type = current_type

    return spans


def entity_prf(gold_sequences, pred_sequences):
    gold_all = set()
    pred_all = set()

    for sent_id, (gold_tags, pred_tags) in enumerate(zip(gold_sequences, pred_sequences)):
        gold_spans = bio_to_spans(gold_tags)
        pred_spans = bio_to_spans(pred_tags)

        # Add sentence ID so identical spans in different sentences are distinct.
        gold_all.update((sent_id, *span) for span in gold_spans)
        pred_all.update((sent_id, *span) for span in pred_spans)

    correct = len(gold_all & pred_all)
    precision = correct / len(pred_all) if pred_all else 0.0
    recall = correct / len(gold_all) if gold_all else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    return precision, recall, f1


gold = [["B-PER", "I-PER", "O", "B-LOC", "I-LOC", "I-LOC"]]
pred = [["B-PER", "O", "O", "B-LOC", "I-LOC", "O"]]

precision, recall, f1 = entity_prf(gold, pred)

print("entity precision:", precision)
print("entity recall:", recall)
print("entity F1:", f1)
```

</details>

This stricter metric is why NER scores may look lower than token accuracy. A single boundary mistake can turn an almost-correct entity into an incorrect span.

#### **BIO Tagging Evaluation** {#bio-tagging-evaluation}

BIO tagging is a common format for representing entity spans as token labels.

| Prefix | Meaning | Example |
|---|---|---|
| `B-` | beginning of an entity | `B-PER` for `Barack` |
| `I-` | inside the same entity | `I-PER` for `Obama` |
| `O` | outside any entity | `visited` |

For the sentence:

```text
Barack Obama visited New York City
```

A BIO sequence might be:

```text
Barack  Obama  visited  New    York   City
B-PER   I-PER  O        B-LOC  I-LOC  I-LOC
```

BIO evaluation has two layers. First, the tag sequence must be valid. Second, the decoded entity spans must match the gold spans.

Some invalid or suspicious transitions include:

| Previous Tag | Current Tag | Problem |
|---|---|---|
| `O` | `I-PER` | `I` appears without a preceding `B` |
| `B-LOC` | `I-PER` | entity type changes inside a span |
| `I-ORG` | `I-LOC` | entity type changes inside a span |

Different evaluation scripts handle malformed sequences differently. Some treat an invalid `I-PER` after `O` as `B-PER`. Others count it as an error. This detail can change reported scores, so it is important to know how the evaluation script decodes tags.

<details>
<summary>Python Checking BIO Transition Validity</summary>

```python
def find_bio_errors(tags):
    """Return positions where BIO transitions are invalid or suspicious."""
    errors = []
    previous_type = None
    previous_inside_entity = False

    for i, tag in enumerate(tags):
        if tag == "O":
            previous_type = None
            previous_inside_entity = False
            continue

        prefix, entity_type = tag.split("-", 1)

        if prefix == "B":
            previous_type = entity_type
            previous_inside_entity = True

        elif prefix == "I":
            if not previous_inside_entity:
                errors.append((i, tag, "I tag without an open entity"))
            elif entity_type != previous_type:
                errors.append((i, tag, "I tag type does not match previous entity"))

            previous_type = entity_type
            previous_inside_entity = True

        else:
            errors.append((i, tag, "unknown BIO prefix"))

    return errors


tags = ["O", "I-PER", "O", "B-LOC", "I-PER", "O"]

for error in find_bio_errors(tags):
    print(error)
```

</details>

For NER, BIO tags are usually an encoding format, not the final evaluation target. The final target is the decoded entity span.

#### **Span-Level Matching** {#span-level-matching}

Span-level matching decides whether a predicted entity is counted as correct. The strictest and most common rule is exact match: start boundary, end boundary, and entity type must all match.

```text
Gold: [New York City]LOC
Pred: [New York]LOC
Result: incorrect under exact span match
```

Even though the prediction is close, it misses `City`, so it is not an exact match.

Different tasks may use different matching rules:

| Matching Rule | Correct If | Common Use |
|---|---|---|
| exact match | start, end, and type all match | standard NER benchmarks |
| boundary-only match | start and end match, type ignored | entity detection analysis |
| type-only match | type appears somewhere, boundary less strict | rough diagnostic only |
| partial overlap | predicted span overlaps gold span | some information extraction settings |
| relaxed match | minor boundary differences allowed | user-facing extraction evaluation |

Exact match is strict but clean. Partial match is more forgiving but can hide boundary problems. For research benchmarks, exact entity-level F1 is usually preferred because it makes systems comparable.

<details>
<summary>Python Exact vs Partial Span Matching</summary>

```python
def overlaps(span_a, span_b):
    """Check whether two spans overlap.

    Each span is (start, end, type), where end is exclusive.
    """
    start_a, end_a, type_a = span_a
    start_b, end_b, type_b = span_b

    same_type = type_a == type_b
    has_overlap = start_a < end_b and start_b < end_a

    return same_type and has_overlap


gold_span = (3, 6, "LOC")   # New York City
pred_exact = (3, 6, "LOC")  # New York City
pred_partial = (3, 5, "LOC") # New York
pred_wrong_type = (3, 6, "ORG")

print("exact match:", pred_exact == gold_span)
print("partial overlap:", overlaps(pred_partial, gold_span))
print("wrong type exact:", pred_wrong_type == gold_span)
print("wrong type overlap:", overlaps(pred_wrong_type, gold_span))
```

</details>

When reporting sequence labeling results, always state whether the metric is token-level, exact span-level, or relaxed span-level.

#### **Common Evaluation Mistakes in NER** {#common-evaluation-mistakes-in-ner}

NER evaluation has several common traps.

The first mistake is reporting token accuracy as if it were entity performance. Since `O` tokens dominate, token accuracy can be high even when the model extracts entities poorly.

The second mistake is ignoring subword tokenization. Transformer tokenizers may split words into subtokens:

```text
Washington -> Washington
unaffordable -> una ##ff ##ordable
```

If labels are assigned to words but predictions are produced for subtokens, evaluation must align them carefully. Usually, only the first subtoken is evaluated, and the remaining subtokens are ignored with `-100` during training and evaluation.

The third mistake is mixing label schemes. BIO, BIOES, and BILOU are related but not identical. A model trained with BIOES should be evaluated with a decoder that understands `E` and `S` tags.

The fourth mistake is evaluating only overall F1. Entity types may behave very differently. A model may perform well on common `PER` and `LOC` entities but fail on rare biomedical entities, product names, or legal references.

<details>
<summary>Python Aligning Word Labels to Subword Tokens</summary>

```python
# This example assumes a Hugging Face fast tokenizer output with word_ids().
# The goal is to label only the first subtoken of each word.

def align_labels_with_subwords(word_labels, word_ids):
    aligned_labels = []
    previous_word_id = None

    for word_id in word_ids:
        if word_id is None:
            # Special tokens like [CLS] and [SEP] should be ignored.
            aligned_labels.append(-100)
        elif word_id != previous_word_id:
            # First subtoken of a word receives the original word label.
            aligned_labels.append(word_labels[word_id])
        else:
            # Later subtokens are ignored for loss and evaluation.
            aligned_labels.append(-100)

        previous_word_id = word_id

    return aligned_labels

# Example:
# tokens:     [CLS], John, lives, in, New, York, [SEP]
# word_ids:   None, 0,    1,     2,  3,   4,    None
word_labels = ["B-PER", "O", "O", "B-LOC", "I-LOC"]
word_ids = [None, 0, 1, 2, 3, 4, None]

print(align_labels_with_subwords(word_labels, word_ids))
```

</details>

A practical NER evaluation report should include:

| Report Item | Why It Matters |
|---|---|
| entity-level micro F1 | standard overall benchmark score |
| per-type precision / recall / F1 | shows which entity types fail |
| token accuracy | useful sanity check, not final metric |
| boundary error examples | reveals span mistakes |
| confusion between entity types | shows semantic classification errors |
| subword alignment policy | prevents hidden evaluation mismatch |

For sequence labeling tasks, the main lesson is that labels live at the token level, but many applications care about spans. Good evaluation must bridge those two views.

### **Language Modeling Evaluation** {#language-modeling-evaluation}

A language model does not usually return one class label. Given a context, it assigns a probability distribution to the next token, and repeats this operation across a sequence. Its evaluation therefore asks two related questions: **how much probability did the model assign to the observed text?** and **does that probability reflect useful language behavior?**

For an autoregressive language model, a token sequence $x_1, x_2, \ldots, x_T$ is factorized as:

$$
P(x_1, x_2, \ldots, x_T)
= \prod_{t=1}^{T} P(x_t \mid x_{<t})
$$

Here, $T$ is the number of predicted tokens, $x_t$ is the target token at position $t$, and $x_{<t}$ denotes all tokens before it. The model is evaluated one prediction at a time: after reading the previous context, how much probability did it assign to the token that actually appeared next?

Suppose the sentence is `the cat sat down`. The corresponding evaluation events may be:

| Context | Correct next token | Probability assigned to the correct token |
|---|---:|---:|
| `<BOS>` | `the` | 0.50 |
| `the` | `cat` | 0.40 |
| `the cat` | `sat` | 0.25 |
| `the cat sat` | `down` | 0.80 |

The probability of the entire sequence is the product $0.50 \times 0.40 \times 0.25 \times 0.80 = 0.04$. Products of many probabilities become extremely small, so practical evaluation works in log space and averages over tokens.

#### **Negative Log-Likelihood** {#negative-log-likelihood}

Negative Log-Likelihood (NLL) measures how surprised the model is by the reference tokens. For one sequence, it is:

$$
\operatorname{NLL}(x_{1:T})
= -\sum_{t=1}^{T}\log P(x_t \mid x_{<t})
$$

$P(x_t \mid x_{<t})$ is the probability assigned to the correct next token. The logarithm converts the product of token probabilities into a sum, which is numerically stable and easier to optimize. The minus sign makes the score non-negative because the logarithm of a probability between 0 and 1 is non-positive.

The behavior of NLL is important:

- If the model gives the correct token a high probability, $-\log(p)$ is small.
- If it gives the correct token a low probability, $-\log(p)$ becomes large.
- A confidently wrong prediction is penalized much more than a mildly uncertain one.

For example, using the natural logarithm, assigning probability $0.8$ gives a loss of about $0.223$, while assigning probability $0.01$ gives about $4.605$. NLL therefore rewards calibrated confidence rather than merely checking whether the highest-probability token was correct.

<details>
<summary>Python Computing Token and Sequence NLL</summary>

```python
import math

# Probabilities assigned to the correct next token at each position.
target_probabilities = [0.50, 0.40, 0.25, 0.80]

# Step 1: convert each probability into token-level negative log-likelihood.
token_nll = [-math.log(probability) for probability in target_probabilities]

# Step 2: sum token losses to obtain sequence NLL.
sequence_nll = sum(token_nll)

# Step 3: average by token count so sequences of different lengths are comparable.
mean_nll = sequence_nll / len(token_nll)

print("Token NLL:", [round(value, 3) for value in token_nll])
print("Sequence NLL:", round(sequence_nll, 3))
print("Mean token NLL:", round(mean_nll, 3))
```

</details>

Sequence NLL grows with sequence length because it adds one term per predicted token. When comparing examples or datasets of different sizes, report the **mean NLL per non-padding target token**, not only the total.

#### **Cross-Entropy** {#cross-entropy}

Cross-entropy compares the reference token distribution with the probability distribution predicted by the model. At position $t$, let $y_{t,v}$ indicate whether vocabulary token $v$ is the correct token, and let $p_{t,v}$ be the model's predicted probability for that token:

$$
H_t(y,p) = -\sum_{v=1}^{|V|} y_{t,v}\log p_{t,v}
$$

$|V|$ is the vocabulary size. For a standard one-hot target, only the correct token has $y_{t,v}=1$; every other target entry is zero. The sum therefore reduces to:

$$
H_t(y,p) = -\log p_{t,\text{correct}}
$$

This is why token-level cross-entropy and token-level NLL are numerically the same in ordinary language-model evaluation. The terms emphasize different viewpoints: **NLL** describes the likelihood of the observed data under the model, while **cross-entropy** describes the disagreement between the target and predicted distributions.

In a padded mini-batch, the usual corpus-level score is:

$$
\mathcal{L}_{\text{CE}}
= -\frac{1}{N}\sum_{i=1}^{B}\sum_{t=1}^{T_i}
m_{i,t}\log P(x_{i,t}\mid x_{i,<t})
$$

$B$ is the batch size, $T_i$ is the padded length of sequence $i$, $m_{i,t}$ is 1 for a real target token and 0 for padding, and $N=\sum_{i,t}m_{i,t}$ is the total number of evaluated tokens. Masking matters: if padding tokens enter the denominator, the score no longer represents the model's behavior on real language.

<details>
<summary>PyTorch Masked Cross-Entropy</summary>

```python
import torch
import torch.nn.functional as F

# Two sequences, three positions, and a vocabulary of four tokens.
# logits[b, t, v] contains an unnormalized score for vocabulary token v.
logits = torch.tensor([
    [[3.0, 1.0, 0.0, -1.0], [0.2, 2.4, 0.1, -0.5], [0.1, 0.2, 2.0, 0.0]],
    [[1.5, 0.2, 0.1, -0.4], [0.1, 0.3, 0.2, 2.2], [0.0, 0.0, 0.0, 0.0]],
])

# The final target in the second sequence is padding, represented by -100.
targets = torch.tensor([
    [0, 1, 2],
    [0, 3, -100],
])

# Flatten batch and time dimensions. ignore_index excludes padding from both
# the loss sum and the number of evaluated tokens.
mean_cross_entropy = F.cross_entropy(
    logits.reshape(-1, logits.size(-1)),
    targets.reshape(-1),
    ignore_index=-100,
    reduction="mean",
)

print("Mean cross-entropy:", mean_cross_entropy.item())
```

</details>

For causal language models, logits and labels must also be aligned correctly: logits produced at position $t$ predict the token at position $t+1$. Many model libraries perform this shift internally, but a custom evaluation loop must verify it explicitly.

#### **Perplexity** {#perplexity}

Perplexity (PPL) is the exponential of the mean token NLL, or equivalently the exponential of cross-entropy when natural logarithms are used:

$$
\operatorname{PPL}
= \exp\left(
-\frac{1}{N}\sum_{t=1}^{N}\log P(x_t\mid x_{<t})
\right)
= \exp(\mathcal{L}_{\text{CE}})
$$

$N$ is the number of evaluated target tokens and $\mathcal{L}_{\text{CE}}$ is their mean cross-entropy. Lower perplexity is better. A useful intuition is that perplexity approximates the model's **effective number of plausible choices** at each position. A perplexity of 10 can be read loosely as the model being as uncertain as if it had to choose among about 10 equally likely tokens. This is an intuition, not a literal statement that exactly 10 tokens always have equal probability.

For the earlier probabilities, the mean NLL is about $0.805$, so:

$$
\operatorname{PPL}=e^{0.805}\approx 2.24
$$

<details>
<summary>Python Computing Perplexity</summary>

```python
import math

probabilities = [0.50, 0.40, 0.25, 0.80]
mean_nll = sum(-math.log(p) for p in probabilities) / len(probabilities)
perplexity = math.exp(mean_nll)

print(f"Mean NLL: {mean_nll:.3f}")
print(f"Perplexity: {perplexity:.3f}")
```

</details>

The following plot makes the relationship visible. Because perplexity is exponential, a modest increase in cross-entropy can represent a much larger increase in uncertainty.

<details>
<summary>Python Plotting Cross-Entropy and Perplexity</summary>

```python
import numpy as np
import matplotlib.pyplot as plt

cross_entropy = np.linspace(0, 5, 200)
perplexity = np.exp(cross_entropy)

plt.figure(figsize=(7, 4))
plt.plot(cross_entropy, perplexity, color="teal", linewidth=2)
plt.xlabel("Mean cross-entropy / NLL")
plt.ylabel("Perplexity")
plt.title("Perplexity grows exponentially with cross-entropy")
plt.grid(alpha=0.25)
plt.show()
```

</details>

Perplexity comparisons are valid only when the evaluation setup is aligned. Two models should use the same test corpus, preprocessing, context policy, and preferably the same tokenization units. A tokenizer that splits text into fewer tokens can change per-token perplexity even when the underlying text prediction quality has not improved. For comparisons across tokenizers, byte- or character-normalized measures such as **bits per byte** can be more meaningful.

#### **Token-Level vs Sequence-Level Evaluation** {#token-level-vs-sequence-level-evaluation}

Token-level evaluation asks how well the model predicts individual target positions. It usually averages NLL across all non-padding tokens in the corpus:

$$
\mathcal{L}_{\text{token}}
= \frac{\sum_{i=1}^{M}\sum_{t=1}^{T_i}\ell_{i,t}}
{\sum_{i=1}^{M}T_i}
$$

$M$ is the number of sequences, $T_i$ is the evaluated length of sequence $i$, and $\ell_{i,t}$ is the NLL for token $t$. Every token receives equal weight, so long documents contribute more than short ones. This is the standard aggregation for corpus perplexity.

Sequence-level evaluation first summarizes each sequence and then averages across sequences:

$$
\mathcal{L}_{\text{sequence}}
= \frac{1}{M}\sum_{i=1}^{M}
\left(\frac{1}{T_i}\sum_{t=1}^{T_i}\ell_{i,t}\right)
$$

Here, every sequence receives equal weight regardless of length. This can be useful when each document, prompt, or user conversation is one meaningful evaluation unit. The two aggregations differ whenever sequence lengths differ.

<details>
<summary>Python Comparing Token- and Sequence-Level Aggregation</summary>

```python
import numpy as np

# Per-token NLL values for one short and one long document.
document_losses = [
    [0.2, 0.3],                    # short, easy document
    [1.1, 1.0, 0.9, 1.2, 1.0],   # long, difficult document
]

# Token-level macro over all tokens: long documents have more influence.
token_weighted_nll = np.mean([loss for doc in document_losses for loss in doc])

# Sequence-level macro: each document contributes one mean value.
per_document_nll = [np.mean(doc) for doc in document_losses]
sequence_weighted_nll = np.mean(per_document_nll)

print("Token-weighted NLL:", round(token_weighted_nll, 3))
print("Sequence-weighted NLL:", round(sequence_weighted_nll, 3))
print("Per-document NLL:", [round(x, 3) for x in per_document_nll])
```

</details>

Sequence probability itself is usually a poor raw comparison because multiplying more probabilities automatically makes longer sequences less likely. Length-normalized log-likelihood is preferable when ranking candidate sentences of different lengths, although generation systems may still need an explicit length penalty to avoid favoring unnaturally short outputs.

#### **Why Perplexity Is Not Enough** {#why-perplexity-is-not-enough}

Perplexity is useful because it is automatic, reproducible, and closely connected to the language-model training objective. It can reveal whether a model predicts in-domain text better after architectural changes, optimization changes, or additional pretraining. However, it measures probability assignment to a fixed reference corpus, not the complete quality of generated language.

Several limitations matter in practice:

- **Tokenizer dependence:** word-, subword-, character-, and byte-level models create different prediction units, so their raw perplexities are not automatically comparable.
- **Domain dependence:** a biomedical model may have low perplexity on clinical notes and high perplexity on social media. This reflects specialization, not necessarily general inferiority.
- **Reference dependence:** valid alternative continuations receive no credit when evaluation asks only for the probability of one observed continuation.
- **Weak connection to factuality and safety:** fluent misinformation, toxic continuations, and memorized private text can all receive high probability.
- **No direct task guarantee:** lower perplexity does not always produce better summarization, question answering, dialogue, or retrieval-augmented generation.
- **Context-window sensitivity:** truncating context can increase perplexity, while evaluation with overlapping windows can accidentally score some tokens with more context than others.

A stronger evaluation combines intrinsic likelihood metrics with behavior-oriented checks:

| Evaluation question | Suitable evidence |
|---|---|
| Does the model assign probability well to held-out text? | Mean NLL, cross-entropy, perplexity |
| Is the comparison fair across tokenizers? | Same tokenizer or bits per byte/character |
| Does generated text match a reference? | BLEU, ROUGE, BERTScore, task-specific metrics |
| Is the output useful and coherent? | Human evaluation or carefully validated model-based judging |
| Is the output factual and safe? | Factuality tests, safety suites, retrieval-grounding checks |
| Where does the model fail? | Per-domain, per-length, per-language, and per-token-frequency slices |

The practical takeaway is that **NLL is the additive penalty, cross-entropy is its average distributional interpretation, and perplexity is the same information expressed on an intuitive exponential scale**. Use them for controlled likelihood comparisons, but pair them with downstream, robustness, safety, and human-centered evaluation before claiming that one language model is better overall.

### **Text Generation Evaluation** {#text-generation-evaluation}

Text generation evaluation is more difficult than classification evaluation because a prompt rarely has only one correct output. The reference summary `The storm delayed several flights` and the candidate `Several flights were postponed because of the storm` express almost the same meaning while sharing relatively few surface forms. Conversely, a candidate can copy many reference words while reversing a fact, omitting a critical condition, or producing an incoherent sentence.

The object being evaluated is usually a triple:

```text
source or prompt + candidate generation + one or more references
```

The source supplies task context, the candidate is the model output, and a reference is one acceptable human-written answer. Not every metric uses all three. BLEU, ROUGE, METEOR, and BERTScore mainly compare candidate and reference; an LLM judge or human evaluator can also inspect the source and assess factual consistency, instruction following, style, and safety.

A complete generation evaluation should distinguish at least four dimensions:

| Dimension | Question |
|---|---|
| surface overlap | does the candidate reuse the same words or phrases as the reference? |
| semantic similarity | does it communicate approximately the same meaning? |
| task correctness | does it satisfy the prompt and preserve facts from the source? |
| linguistic quality | is it coherent, fluent, concise, and appropriate for the audience? |

No single metric measures all four dimensions reliably. The metrics below should therefore be understood as different measurement tools, not as interchangeable definitions of generation quality.

#### **BLEU** {#bleu}

BLEU (Bilingual Evaluation Understudy) was designed for machine translation. Its central idea is **modified n-gram precision**: a good candidate should contain many short word sequences that also appear in one or more reference translations, but repeated words should not receive unlimited credit.

For n-grams of order $n$, modified precision is:

$$
p_n =
\frac{
\sum_{g \in G_n(c)} \min\left(\operatorname{count}_c(g),\operatorname{maxrefcount}(g)\right)
}{
\sum_{g \in G_n(c)} \operatorname{count}_c(g)
}
$$

$c$ is the candidate sentence, $G_n(c)$ is the set of candidate n-grams of length $n$, and $g$ is one particular n-gram. $\operatorname{count}_c(g)$ counts how often it occurs in the candidate, while $\operatorname{maxrefcount}(g)$ is its largest count among the references. The `min` operation clips repeated candidate n-grams to the amount supported by a reference.

For example, if the candidate repeats `the` seven times but the reference contains `the` twice, only two occurrences are counted as matched. This prevents a nonsensical repetition from receiving perfect unigram precision.

Precision alone would reward very short candidates. The candidate `the cat` may match two words perfectly while failing to translate the rest of a sentence. BLEU therefore introduces a brevity penalty:

$$
BP =
\begin{cases}
1, & c > r \\
\exp(1-r/c), & c \le r
\end{cases}
$$

Here, $c$ is total candidate length and $r$ is the effective reference length. If the candidate is at least as long as the reference, $BP=1$. If it is too short, $BP$ decreases exponentially.

The final score combines n-gram precisions with a weighted geometric mean:

$$
\operatorname{BLEU}
= BP \cdot \exp\left(\sum_{n=1}^{N}w_n\log p_n\right)
$$

$N$ is the maximum n-gram order, commonly 4, and $w_n$ is the weight assigned to each order, commonly $1/4$. The geometric mean makes BLEU strict: if an unsmoothed sentence has no matching 4-gram, $p_4=0$ and the entire sentence score can collapse to zero. This is why BLEU is more stable at **corpus level**, while sentence-level BLEU needs an explicitly reported smoothing method.

<details>
<summary>Python Corpus BLEU with SacreBLEU</summary>

```python
import sacrebleu

# Each candidate must align with the reference at the same list position.
candidates = [
    "Several flights were delayed by the storm.",
    "The committee approved the proposal.",
]

# SacreBLEU expects references grouped by reference set.
# Add another inner list when multiple valid references are available.
references = [[
    "The storm delayed several flights.",
    "The proposal was approved by the committee.",
]]

# Corpus BLEU aggregates n-gram counts before computing the final score.
result = sacrebleu.corpus_bleu(candidates, references)

print(f"BLEU: {result.score:.2f}")
print("Precisions for 1- to 4-grams:", result.precisions)
print("Brevity penalty:", result.bp)
print("Reproducible signature:", result)
```

</details>

BLEU is useful when phrasing is relatively constrained, especially machine translation with multiple references and corpus-level reporting. It is much less reliable for dialogue, creative writing, or abstractive summarization, where a valid response may use entirely different wording. A high BLEU score also does not guarantee factual correctness: changing `increased` to `decreased` affects only one token but reverses the meaning.

#### **ROUGE** {#rouge}

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) was developed for summarization. Whereas BLEU asks how much of the candidate is supported by the reference, ROUGE traditionally emphasizes how much reference content the candidate recovered.

For ROUGE-$N$, n-gram recall is:

$$
R_{\text{ROUGE-}N}
= \frac{\sum_{g \in G_n(r)}\min(\operatorname{count}_c(g),\operatorname{count}_r(g))}
{\sum_{g \in G_n(r)}\operatorname{count}_r(g)}
$$

$r$ is the reference, $c$ is the candidate, and $g$ is a reference n-gram. The numerator counts clipped overlap and the denominator counts all reference n-grams. ROUGE-1 measures unigram overlap and mostly reflects content-word coverage. ROUGE-2 measures bigram overlap and is more sensitive to local phrase structure.

ROUGE-L uses the **Longest Common Subsequence** (LCS). An LCS preserves token order but does not require matched tokens to be adjacent. If $LCS(c,r)$ is the LCS length, then:

$$
P_{LCS}=\frac{LCS(c,r)}{|c|},
\qquad
R_{LCS}=\frac{LCS(c,r)}{|r|}
$$

$|c|$ and $|r|$ are candidate and reference lengths. Their harmonic combination produces ROUGE-L F1. It rewards in-order content coverage more flexibly than exact bigram matching.

Consider a reference `the storm delayed several flights`:

- `the storm delayed flights` has high recall and preserves the main meaning.
- `the storm delayed several flights and closed every airport in Europe` may also have high ROUGE recall, even though it adds an unsupported claim.

This exposes ROUGE's main limitation: recall-oriented overlap can reward coverage without adequately penalizing hallucinated additions.

<details>
<summary>Python ROUGE-1, ROUGE-2, and ROUGE-L</summary>

```python
from rouge_score import rouge_scorer

reference = "The storm delayed several flights."
candidates = {
    "faithful paraphrase": "Several flights were delayed by the storm.",
    "incomplete": "The storm delayed flights.",
    "unsupported addition": (
        "The storm delayed several flights and closed every airport in Europe."
    ),
}

# use_stemmer lets related forms such as "delay" and "delayed" match.
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True,
)

for name, candidate in candidates.items():
    scores = scorer.score(reference, candidate)
    print(f"\n{name}")
    for metric, score in scores.items():
        print(
            metric,
            f"precision={score.precision:.3f}",
            f"recall={score.recall:.3f}",
            f"f1={score.fmeasure:.3f}",
        )
```

</details>

For summarization, report which ROUGE variants, tokenizer, stemming policy, aggregation method, and implementation were used. `ROUGE-1/2/L F1` is often more balanced than reporting recall alone, but none of these scores directly verifies whether the summary is entailed by the source document.

#### **METEOR** {#meteor}

METEOR was introduced for machine translation to address some weaknesses of exact n-gram overlap. It builds an alignment between candidate and reference words using exact matches and, depending on the implementation and language resources, stem or synonym matches. It then combines unigram precision, unigram recall, and a penalty for fragmented word order.

If $m$ aligned unigrams are found, candidate length is $|c|$, and reference length is $|r|$, then:

$$
P=\frac{m}{|c|},
\qquad
R=\frac{m}{|r|}
$$

A commonly presented METEOR formulation gives recall more weight:

$$
F_{mean}=\frac{10PR}{R+9P}
$$

The alignment is divided into the smallest number of contiguous chunks. If matched words appear in the same order and close together, only a few chunks are needed. If they are heavily reordered, the chunk count rises. A common fragmentation penalty is:

$$
Penalty = \gamma\left(\frac{ch}{m}\right)^\beta
$$

$ch$ is the number of matched chunks, $m$ is the number of matched unigrams, and $\gamma$ and $\beta$ control penalty strength. The final score is:

$$
\operatorname{METEOR}=(1-Penalty)F_{mean}
$$

The exact parameters and matching modules vary across METEOR versions, so results from different implementations should not be silently mixed. The intuition remains stable: reward content matches, allow limited linguistic variation, and penalize scattered or badly reordered matches.

<details>
<summary>Python Sentence-Level METEOR</summary>

```python
import nltk
from nltk.translate.meteor_score import meteor_score

# Run once if WordNet resources are not already installed:
# nltk.download("wordnet")
# nltk.download("omw-1.4")

reference = "The storm delayed several flights".lower().split()
candidates = {
    "close wording": "The storm delayed many flights".lower().split(),
    "paraphrase": "Several flights were postponed by the storm".lower().split(),
    "poor order": "Flights storm the several delayed".lower().split(),
}

for name, candidate in candidates.items():
    # NLTK expects pre-tokenized references and hypothesis.
    score = meteor_score([reference], candidate)
    print(f"{name:15s}: {score:.3f}")
```

</details>

METEOR is often more informative than BLEU for individual sentences because it considers recall and flexible token matching. Its disadvantages are greater language-resource dependence, implementation variability, and continued reliance on word alignment rather than full contextual meaning.

#### **BERTScore** {#bertscore}

BERTScore replaces exact n-gram matching with **contextual token similarity**. A pretrained encoder maps every reference and candidate token to an embedding. The metric computes cosine similarity between every candidate-reference token pair, then greedily gives each token credit for its most similar counterpart.

![BERTScore token matching process](assets/bertscore-token-matching.png){fig-alt="BERTScore encodes reference and candidate tokens, computes pairwise cosine similarities, selects maximum token matches, and applies IDF importance weights." width=100%}

*Source: [official BERTScore repository](https://github.com/Tiiiger/bert_score). The figure illustrates contextual token matching and optional IDF weighting.*

Let $\hat{x}_i$ be the contextual embedding of candidate token $i$, and $x_j$ the embedding of reference token $j$. With cosine similarity $\cos(\hat{x}_i,x_j)$, BERTScore precision and recall are:

$$
P_{BERT}=\frac{1}{|c|}\sum_{i=1}^{|c|}\max_j \cos(\hat{x}_i,x_j)
$$

$$
R_{BERT}=\frac{1}{|r|}\sum_{j=1}^{|r|}\max_i \cos(x_j,\hat{x}_i)
$$

$P_{BERT}$ asks whether each candidate token has a semantically similar reference token. $R_{BERT}$ asks whether each reference token is represented somewhere in the candidate. Their harmonic mean is:

$$
F_{BERT}=2\frac{P_{BERT}R_{BERT}}{P_{BERT}+R_{BERT}}
$$

Optional inverse document frequency (IDF) weighting gives more influence to informative tokens than to frequent function words. Because embeddings are contextual, `bank` in `river bank` is compared differently from `bank` in `bank account`.

<details>
<summary>Python BERTScore for Semantic Generation Evaluation</summary>

```python
from bert_score import score

candidates = [
    "Several flights were postponed because of the storm.",
    "The weather was pleasant and all flights left early.",
]
references = [
    "The storm delayed several flights.",
    "The storm delayed several flights.",
]

# The scorer returns one precision, recall, and F1 value per example.
# rescale_with_baseline makes scores easier to interpret for the chosen model.
precision, recall, f1 = score(
    candidates,
    references,
    lang="en",
    model_type="microsoft/deberta-large-mnli",
    rescale_with_baseline=True,
    verbose=False,
)

for candidate, p, r, f in zip(candidates, precision, recall, f1):
    print("\nCandidate:", candidate)
    print(f"BERTScore P={p:.3f}, R={r:.3f}, F1={f:.3f}")
```

</details>

BERTScore handles paraphrases better than BLEU or ROUGE, but semantic similarity is not the same as factual equivalence. Antonyms, negation, numbers, and named entities can still be embedded closely. The encoder model, layer, language setting, baseline rescaling, and package version must be reported because they influence the score.

#### **Embedding-Based Similarity** {#embedding-based-similarity}

Embedding-based similarity represents the entire candidate and reference as one vector each, then compares those vectors with cosine similarity:

$$
\operatorname{sim}(c,r)
= \cos(e_c,e_r)
= \frac{e_c \cdot e_r}{\lVert e_c\rVert_2\lVert e_r\rVert_2}
$$

$e_c$ and $e_r$ are sentence or document embeddings. The numerator is their dot product, and each denominator term is the Euclidean length of one vector. Cosine similarity measures direction rather than raw magnitude: vectors pointing in a similar semantic direction receive a score closer to 1.

This method is computationally simple after embeddings are produced and is useful for paraphrase-style comparisons, answer matching, clustering evaluation, and large-scale screening. Its granularity differs from BERTScore:

| Method | Matching unit | Main behavior |
|---|---|---|
| BERTScore | contextual token to token | exposes which content tokens align or are missing |
| sentence embedding similarity | one vector per complete text | compresses overall meaning into a single global comparison |

Global compression can hide local errors. `Revenue increased from 10 to 20 million` and `Revenue decreased from 20 to 10 million` contain nearly identical concepts and may have high embedding similarity despite stating opposite trends.

<details>
<summary>Python Sentence-Embedding Similarity</summary>

```python
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")

reference = "The storm delayed several flights."
candidates = [
    "Several flights were postponed because of the storm.",
    "The storm caused several flights to leave early.",
    "The chef prepared dinner in the kitchen.",
]

# Step 1: encode the reference and candidates in the same embedding space.
embeddings = model.encode([reference] + candidates, normalize_embeddings=True)
reference_embedding = embeddings[0:1]
candidate_embeddings = embeddings[1:]

# Step 2: compare every candidate with the reference.
similarities = cosine_similarity(candidate_embeddings, reference_embedding).ravel()

# Step 3: inspect examples instead of treating one threshold as universally valid.
for candidate, similarity in zip(candidates, similarities):
    print(f"{similarity:.3f} | {candidate}")
```

</details>

Embedding similarity is best treated as one semantic signal. Before using a cutoff, calibrate it on task-specific human labels and inspect contrastive examples involving negation, quantities, entities, and causal relations.

#### **LLM-as-a-Judge** {#llm-as-a-judge}

LLM-as-a-Judge uses a language model to evaluate another model's output according to a written rubric. It is useful when quality depends on dimensions that simple overlap cannot capture, such as instruction following, factual consistency with a supplied source, reasoning quality, style, or safety.

Three common formulations are:

| Formulation | Judge input | Output | Typical use |
|---|---|---|---|
| pointwise | prompt, one response, rubric | score or category | absolute quality monitoring |
| pairwise | prompt, response A, response B, rubric | preferred response or tie | model comparison |
| reference-based | prompt, response, reference, rubric | score plus explanation | tasks with trusted answers |

A reliable pipeline has several explicit stages:

```text
define criterion -> write rubric -> randomize response order -> judge independently
-> validate output schema -> aggregate repeated judgments -> compare with human labels
```

The rubric must separate criteria. Asking for one vague `overall quality` score allows fluency to conceal factual errors. A grounded question-answering rubric might independently score correctness, completeness, citation support, and relevance.

<details>
<summary>Python Building a Structured Pairwise Judge Prompt</summary>

```python
import json
import random

def build_pairwise_judge_prompt(question, response_a, response_b):
    """Randomize response order and request a machine-readable judgment."""
    responses = [("A", response_a), ("B", response_b)]
    random.shuffle(responses)

    prompt = f"""
You are evaluating two answers to the same user question.

Question:
{question}

Response {responses[0][0]}:
{responses[0][1]}

Response {responses[1][0]}:
{responses[1][1]}

Evaluate correctness, relevance, completeness, and unsupported claims.
Do not prefer an answer because it is longer or appears first.
Return JSON only:
{{
  "winner": "A", "B", or "tie",
  "correctness_a": integer from 1 to 5,
  "correctness_b": integer from 1 to 5,
  "reason": "brief evidence-based explanation"
}}
""".strip()

    # Keep the mapping so the randomized label can be converted back to model ID.
    label_to_original = {
        responses[0][0]: responses[0][1],
        responses[1][0]: responses[1][1],
    }
    return prompt, label_to_original


prompt, mapping = build_pairwise_judge_prompt(
    question="What caused the flight delays?",
    response_a="The storm caused the delays.",
    response_b="All flights departed early because the weather was clear.",
)

print(prompt)

# In a real pipeline, pass `prompt` to the chosen judge model, parse the JSON,
# validate allowed score ranges, and store judge model/version with the result.
```

</details>

LLM judges scale better than manual review, but they are not neutral ground truth. Common failure modes include position bias, verbosity bias, style bias, self-preference, sensitivity to prompt wording, and weak handling of domain expertise. Pairwise order should be swapped or randomized; judgments should be repeated when variance matters; and agreement with expert human ratings should be measured on a representative calibration set. Judge model name, version, prompt, decoding settings, and rubric are part of the metric definition and should be versioned.

#### **Human Evaluation for Generation** {#human-evaluation-for-generation}

Human evaluation remains essential when acceptable outputs are diverse or when the target property is difficult to automate. Humans can inspect whether a summary preserves the source meaning, whether a dialogue response is genuinely helpful, or whether a medical answer contains a subtle but dangerous claim.

Human evaluation is only useful when the protocol is well defined. A sound study specifies:

- **unit:** sentence, document, dialogue turn, or complete conversation;
- **criteria:** factuality, relevance, fluency, completeness, safety, or another observable property;
- **scale:** binary labels, Likert ratings, error spans, or pairwise preference;
- **annotators:** target users, trained general annotators, or domain experts;
- **blinding and randomization:** hide model identity and randomize response order;
- **quality control:** examples, qualification tasks, repeated items, and disagreement resolution;
- **uncertainty:** confidence intervals and inter-annotator agreement, not only a mean score.

Pairwise preference is often easier and more reliable than assigning an absolute score because the evaluator answers a concrete question: which of these two responses better satisfies the same rubric? Absolute scores remain useful when a fixed quality threshold matters, but annotators may interpret a `4 out of 5` differently.

<details>
<summary>Python Aggregating Human Pairwise Preferences</summary>

```python
import pandas as pd
from sklearn.metrics import cohen_kappa_score

# Each row is one judgment. Two annotators evaluate the same item independently.
ratings = pd.DataFrame({
    "item_id": [1, 1, 2, 2, 3, 3, 4, 4],
    "annotator": ["ann_1", "ann_2"] * 4,
    "winner": ["A", "A", "B", "A", "tie", "tie", "B", "B"],
})

# Step 1: convert to one row per item for agreement analysis.
agreement_table = ratings.pivot(
    index="item_id",
    columns="annotator",
    values="winner",
)

# Step 2: measure agreement beyond raw matching.
kappa = cohen_kappa_score(
    agreement_table["ann_1"],
    agreement_table["ann_2"],
)

# Step 3: aggregate preference counts, keeping ties visible.
preference_rate = ratings["winner"].value_counts(normalize=True).sort_index()

print("Agreement table:\n", agreement_table)
print(f"\nCohen's kappa: {kappa:.3f}")
print("\nPreference proportions:\n", preference_rate)
```

</details>

A low agreement score does not automatically mean annotators performed poorly. It may reveal an ambiguous rubric, inherently subjective examples, missing context, or several valid outputs. Disagreements should be inspected qualitatively before collapsing judgments into one number.

The metrics in this section can now be compared directly:

| Method | Main signal | Strong use case | Important limitation |
|---|---|---|---|
| BLEU | clipped n-gram precision plus brevity penalty | corpus-level machine translation | weak for paraphrases and sentence-level open generation |
| ROUGE | reference content overlap, often recall-oriented | summarization coverage | does not verify source factuality or penalize all additions |
| METEOR | aligned unigram precision/recall plus order penalty | sentence-level translation comparison | resource- and implementation-dependent |
| BERTScore | contextual token alignment | semantic candidate-reference comparison | can miss negation, entity, and numerical errors |
| embedding similarity | global vector similarity | fast semantic screening and paraphrase matching | compresses away local factual differences |
| LLM-as-a-Judge | rubric-based model judgment | instruction following and multi-dimensional quality | bias, drift, cost, and judge validation requirements |
| human evaluation | direct judgment under a protocol | final quality, usefulness, safety, and expert correctness | expensive, slower, and sensitive to study design |

In practice, choose metrics according to the failure you need to detect. A summarization system might report ROUGE for coverage, BERTScore for semantic overlap, a source-grounded factuality check for hallucinations, and blinded human preference for overall usefulness. Agreement among several imperfect signals is much stronger evidence than a small improvement in one automatic metric.

### **Retrieval and Ranking Evaluation** {#retrieval-and-ranking-evaluation}

Retrieval systems do not normally predict one label or generate one sentence. Given a query, they return an **ordered list** of documents, passages, products, answers, or other candidates. Evaluation must therefore measure both **which relevant items were found** and **where they appeared in the ranking**.

This distinction matters in NLP applications. A search engine user may inspect only the first page. A question-answering system may pass only the top five passages to its reader. A RAG system has a limited context window, so placing irrelevant chunks above supporting evidence wastes context and can distract the generator.

Assume the query is:

> What operational problems can severe storms cause for airlines?

The retriever returns five passages with human relevance labels:

| Rank | Passage | Binary relevance | Graded relevance |
|---:|---|---:|---:|
| 1 | storms can delay flights and force cancellations | 1 | 3 |
| 2 | airlines introduced a new loyalty program | 0 | 0 |
| 3 | thunderstorms can temporarily close runways | 1 | 2 |
| 4 | severe weather may require aircraft rerouting | 1 | 1 |
| 5 | airport restaurants changed opening hours | 0 | 0 |

Suppose four relevant passages exist in the whole evaluation collection, meaning one relevant passage was not retrieved in the top five. This single ranking will be reused below. It shows why no one metric is sufficient: precision measures focus, recall measures coverage, MRR emphasizes the first useful result, MAP rewards repeatedly ranking relevant results early, and NDCG also respects degrees of relevance.

Before computing any ranking metric, the evaluation dataset must define:

- the query and its information need;
- the retrieval unit, such as document, passage, or chunk;
- which candidates are relevant to each query;
- whether relevance is binary or graded;
- the cutoff $K$ that reflects the real interface or context budget.

Without reliable relevance judgments, a metric may mark an unjudged but genuinely useful passage as irrelevant. This is especially common in large or changing corpora.

#### **Precision@K** {#precision-k}

Precision@K asks: **among the first $K$ retrieved items, what fraction is relevant?**

$$
\operatorname{Precision@K}
= \frac{1}{K}\sum_{i=1}^{K}\operatorname{rel}_i
$$

$K$ is the ranking cutoff, $i$ is a rank position, and $\operatorname{rel}_i$ is 1 when the item at rank $i$ is relevant and 0 otherwise. The numerator counts relevant results inside the top $K$; the denominator is always $K$.

For the example ranking:

$$
\operatorname{Precision@3}=\frac{1+0+1}{3}=\frac{2}{3}\approx0.667
$$

Two of the top three passages are useful. Precision@K therefore measures **purity or focus**. It is important when users inspect only a small list, when irrelevant results are costly, or when every retrieved chunk consumes an LLM context budget.

Precision@K does not tell us how many relevant items were missed. Returning one perfect passage at $K=1$ gives precision 1.0 even if ten other necessary passages remain undiscovered. It also treats every position within the top $K$ equally: a relevant item at rank 1 contributes exactly as much as one at rank $K$.

<details>
<summary>Python Precision@K for Retrieved Passages</summary>

```python
def precision_at_k(binary_relevance, k):
    """Measure the relevant fraction among the first k retrieved items."""
    if k <= 0:
        raise ValueError("k must be positive")

    # Evaluate exactly the requested cutoff. In a real benchmark, ensure that
    # every query has at least k retrieved candidates before using this form.
    top_k = binary_relevance[:k]
    if len(top_k) < k:
        raise ValueError("The ranking contains fewer than k items")

    return sum(top_k) / k


ranking = [1, 0, 1, 1, 0]

for k in [1, 3, 5]:
    print(f"Precision@{k}: {precision_at_k(ranking, k):.3f}")
```

</details>

Choose $K$ from the product, not for convenience. `Precision@10` may describe a search results page, while `Precision@5` may fit a RAG system that inserts five chunks into its prompt.

#### **Recall@K** {#recall-k}

Recall@K asks: **how much of all relevant information was recovered within the first $K$ results?**

$$
\operatorname{Recall@K}
= \frac{\sum_{i=1}^{K}\operatorname{rel}_i}{|R_q|}
$$

$|R_q|$ is the total number of relevant items for query $q$ in the evaluation collection. The numerator counts relevant items found in the top $K$, while the denominator includes relevant items that were retrieved lower in the ranking or missed entirely.

In the example, four relevant passages exist overall and two appear in the top three:

$$
\operatorname{Recall@3}=\frac{2}{4}=0.5
$$

Recall@K measures **coverage**. It matters when an answer requires several pieces of evidence, such as a legal question involving multiple clauses or a research question that asks for several causes. High precision with low recall can produce a concise context that omits essential evidence.

Unlike precision, recall cannot be calculated correctly unless the benchmark knows how many relevant items exist. In web-scale search or dynamic corpora, exhaustive relevance judgments may be impossible. Benchmarks often use pooled judgments, where annotators label the union of high-ranked results from several retrievers. This is practical but incomplete and may favor systems similar to those used to build the pool.

<details>
<summary>Python Recall@K and Retrieval Coverage</summary>

```python
def recall_at_k(binary_relevance, total_relevant, k):
    """Measure how many known relevant items appear within the top k."""
    if total_relevant <= 0:
        raise ValueError("total_relevant must be positive")

    retrieved_relevant = sum(binary_relevance[:k])
    return retrieved_relevant / total_relevant


ranking = [1, 0, 1, 1, 0]
total_relevant = 4  # Includes one relevant passage missing from the top five.

for k in [1, 3, 5]:
    print(f"Recall@{k}: {recall_at_k(ranking, total_relevant, k):.3f}")
```

</details>

Precision and recall commonly trade off as $K$ increases. Retrieving more items usually improves coverage, but it can introduce more irrelevant context. In RAG, this trade-off interacts with prompt length, latency, and the generator's ability to ignore distractors.

#### **Mean Reciprocal Rank** {#mean-reciprocal-rank}

Reciprocal Rank (RR) focuses only on the position of the **first relevant result**:

$$
\operatorname{RR}_q=\frac{1}{\operatorname{rank}_q}
$$

$\operatorname{rank}_q$ is the rank position of the first relevant item for query $q$. If the first relevant item appears at rank 1, RR is 1. If it appears at rank 2, RR is $1/2$. At rank 5, it is $1/5$. If no relevant result is retrieved within the evaluated list, RR is usually 0.

Mean Reciprocal Rank averages RR across $Q$ queries:

$$
\operatorname{MRR}
= \frac{1}{|Q|}\sum_{q \in Q}\frac{1}{\operatorname{rank}_q}
$$

$Q$ is the query set and $|Q|$ is its size. The reciprocal makes early movement especially valuable: improving rank 2 to rank 1 increases RR by 0.5, while improving rank 10 to rank 9 changes it by only about 0.011.

MRR is well suited to tasks where one correct result is enough, such as FAQ retrieval, navigational search, or retrieving one answer-bearing passage. It ignores every relevant result after the first. A system that retrieves one useful passage first and misses all remaining evidence can still achieve perfect RR for that query.

<details>
<summary>Python Reciprocal Rank and MRR</summary>

```python
def reciprocal_rank(binary_relevance, k=None):
    """Return reciprocal rank of the first relevant item, optionally within k."""
    evaluated = binary_relevance if k is None else binary_relevance[:k]

    for rank, is_relevant in enumerate(evaluated, start=1):
        if is_relevant:
            return 1.0 / rank
    return 0.0


# Three NLP queries: first relevant passages occur at ranks 1, 3, and nowhere.
query_rankings = [
    [1, 0, 1, 0],
    [0, 0, 1, 1],
    [0, 0, 0, 0],
]

rr_scores = [reciprocal_rank(ranking) for ranking in query_rankings]
mrr = sum(rr_scores) / len(rr_scores)

print("Reciprocal ranks:", [round(score, 3) for score in rr_scores])
print(f"MRR: {mrr:.3f}")
```

</details>

When reporting `MRR@K`, explicitly state the cutoff: queries with no relevant result in the top $K$ receive zero even if a relevant item exists later.

#### **Mean Average Precision** {#mean-average-precision}

Average Precision (AP) rewards a ranking for placing **all relevant items early**, not only the first one. Precision is computed at every rank where a relevant item appears, then those precision values are averaged:

$$
\operatorname{AP}_q
= \frac{1}{|R_q|}
\sum_{i=1}^{N}
\operatorname{Precision@i}\cdot\operatorname{rel}_i
$$

$N$ is the evaluated ranking length. $\operatorname{rel}_i$ acts as a switch: precision at rank $i$ contributes only when that item is relevant. $|R_q|$ is the total number of relevant items, including relevant items missed by the ranking. Therefore, missing relevant documents lowers AP.

For binary relevance `[1, 0, 1, 1, 0]` with four relevant passages in total:

$$
\operatorname{AP}
=\frac{1}{4}
\left(
\frac{1}{1}+\frac{2}{3}+\frac{3}{4}
\right)
\approx0.604
$$

The relevant results occur at ranks 1, 3, and 4. Precision is evaluated at those ranks. The fourth relevant passage was missed, so it contributes nothing while remaining in the denominator.

Mean Average Precision (MAP) averages AP over all queries:

$$
\operatorname{MAP}
=\frac{1}{|Q|}\sum_{q\in Q}\operatorname{AP}_q
$$

MAP is useful when each query can have several relevant documents and relevance is binary. It captures both coverage and ordering more fully than MRR. However, it assumes that all relevant items are equally valuable. It also depends strongly on complete relevance judgments.

<details>
<summary>Python Average Precision and MAP from Rankings</summary>

```python
def average_precision(binary_relevance, total_relevant=None, k=None):
    """Compute AP from an ordered binary relevance list."""
    evaluated = binary_relevance if k is None else binary_relevance[:k]
    known_relevant = sum(binary_relevance) if total_relevant is None else total_relevant

    if known_relevant == 0:
        return 0.0

    relevant_seen = 0
    precision_sum = 0.0

    for rank, is_relevant in enumerate(evaluated, start=1):
        if is_relevant:
            relevant_seen += 1
            precision_sum += relevant_seen / rank

    # A missed relevant item remains in this denominator and lowers AP.
    return precision_sum / known_relevant


rankings = [
    ([1, 0, 1, 1, 0], 4),
    ([0, 1, 1, 0, 0], 3),
    ([1, 1, 0, 0, 0], 2),
]

ap_scores = [
    average_precision(ranking, total_relevant)
    for ranking, total_relevant in rankings
]
mean_average_precision = sum(ap_scores) / len(ap_scores)

print("AP per query:", [round(score, 3) for score in ap_scores])
print(f"MAP: {mean_average_precision:.3f}")
```

</details>

Different libraries use different `AP@K` denominators, such as total relevant items or `min(total relevant, K)`. The chosen convention must be documented; otherwise two results called `MAP@10` may not be directly comparable.

#### **NDCG** {#ndcg}

Normalized Discounted Cumulative Gain (NDCG) is designed for **graded relevance**. A passage can be fully answer-bearing, partially useful, topically related, or irrelevant. NDCG gives more gain to highly relevant results and discounts that gain when the result appears lower in the ranking.

A common formulation of Discounted Cumulative Gain at cutoff $K$ is:

$$
\operatorname{DCG@K}
=\sum_{i=1}^{K}
\frac{2^{\operatorname{rel}_i}-1}{\log_2(i+1)}
$$

$\operatorname{rel}_i$ is the graded relevance score at rank $i$. The gain term $2^{\operatorname{rel}_i}-1$ makes a grade-3 item substantially more valuable than a grade-1 item. The denominator $\log_2(i+1)$ is the position discount: rank 1 is not discounted because $\log_2(2)=1$, while lower ranks contribute progressively less.

Raw DCG depends on how many relevant items a query has. It is normalized by the best possible ordering of the same relevance labels, called Ideal DCG:

$$
\operatorname{NDCG@K}
=\frac{\operatorname{DCG@K}}{\operatorname{IDCG@K}}
$$

$\operatorname{IDCG@K}$ is obtained by sorting items from highest to lowest relevance before calculating DCG. NDCG is normally between 0 and 1, and 1 means the ranking is ideal for the available relevance judgments.

For RAG, graded labels can distinguish:

- `3`: directly contains sufficient answer evidence;
- `2`: contains useful supporting evidence;
- `1`: topically related but insufficient;
- `0`: irrelevant.

This is richer than binary precision or recall, but the grade definitions must be concrete enough for annotators to apply consistently.

<details>
<summary>Python DCG and NDCG with Graded Relevance</summary>

```python
import math

def dcg_at_k(graded_relevance, k):
    """Compute DCG with exponential gain and logarithmic rank discount."""
    score = 0.0
    for rank, relevance in enumerate(graded_relevance[:k], start=1):
        gain = (2 ** relevance) - 1
        discount = math.log2(rank + 1)
        score += gain / discount
    return score


def ndcg_at_k(graded_relevance, k):
    """Normalize actual DCG by the best ordering of the same labels."""
    actual_dcg = dcg_at_k(graded_relevance, k)
    ideal_order = sorted(graded_relevance, reverse=True)
    ideal_dcg = dcg_at_k(ideal_order, k)
    return actual_dcg / ideal_dcg if ideal_dcg > 0 else 0.0


ranking_a = [3, 0, 2, 1, 0]
ranking_b = [1, 3, 0, 2, 0]  # Same items, but the strongest evidence is lower.

for name, ranking in {"A": ranking_a, "B": ranking_b}.items():
    print(
        f"Ranking {name}: DCG@5={dcg_at_k(ranking, 5):.3f}, "
        f"NDCG@5={ndcg_at_k(ranking, 5):.3f}"
    )
```

</details>

NDCG is preferable when relevance has meaningful levels and early placement matters. MAP is often easier to interpret for binary relevance; NDCG is more expressive for graded evidence quality.

#### **Retrieval Evaluation in RAG Systems** {#retrieval-evaluation-in-rag-systems}

A Retrieval-Augmented Generation system has at least two connected components:

```text
query -> retriever / reranker -> retrieved context -> generator -> answer
```

An incorrect final answer can result from several different failures:

- the required evidence was never present in the corpus;
- document parsing or chunking separated crucial information;
- the retriever missed the relevant chunk;
- the retriever found it but ranked it below the context cutoff;
- relevant context was mixed with distracting passages;
- the generator ignored good evidence or added unsupported claims.

End-to-end answer accuracy alone cannot distinguish these causes. RAG evaluation should measure the retrieval and generation stages separately, then examine how they interact. Research frameworks such as ARES explicitly distinguish **context relevance**, **answer faithfulness**, and **answer relevance** when evaluating RAG systems.

![ARES evaluation framework for ranking RAG systems](assets/ares-rag-evaluation-framework.png){fig-alt="ARES evaluation framework with synthetic evaluation data generation, preparation of LLM judges, and ranking RAG systems with confidence intervals." width=92%}

*Source: [ARES: An Automated Evaluation Framework for Retrieval-Augmented Generation Systems](https://aclanthology.org/2024.naacl-long.20/), Figure 1.*

The figure shows one rigorous way to scale RAG evaluation: create domain-specific evaluation triples, prepare judges for several criteria, then combine automated judgments with human labels and uncertainty estimates. A production system does not have to reproduce ARES exactly, but it should retain the same separation of evaluation dimensions.

| Component | Evaluation question | Example evidence |
|---|---|---|
| corpus and chunking | does an answer-bearing chunk exist? | corpus coverage, oracle retrieval analysis |
| retrieval | were relevant chunks found in the top $K$? | Recall@K, hit rate, MRR |
| ranking | were the strongest chunks placed early? | Precision@K, MAP, NDCG |
| context quality | is retrieved context focused and sufficient? | context precision/relevance and context recall |
| generation grounding | are answer claims supported by retrieved context? | faithfulness, attribution, citation correctness |
| final answer | does the response answer the query correctly? | answer correctness, relevance, task success, human judgment |

Two ablation tests are especially useful for locating the bottleneck:

1. **Oracle-context generation:** give the generator known gold evidence. If answers remain poor, retrieval is not the main limitation.
2. **Retrieval-only inspection:** ignore generation and measure whether gold evidence reaches the top $K$. If recall is poor, changing prompts or the generator cannot fix missing context.

RAG relevance labels should match the retrieval unit. If the system retrieves chunks, evaluating only document IDs can overestimate success: the correct document may be retrieved while the selected chunk lacks the answer. Conversely, exact chunk-ID matching can underestimate semantically equivalent overlapping chunks. Many serious benchmarks therefore annotate whether a chunk actually supports the answer, not merely whether it came from the expected document.

<details>
<summary>Python Evaluating Retrieval and Grounding for RAG</summary>

```python
def evaluate_rag_example(
    retrieved_chunk_ids,
    gold_chunk_ids,
    claim_support_labels,
    k,
):
    """Evaluate retrieval coverage and answer grounding for one RAG example.

    claim_support_labels contains one Boolean per atomic answer claim, produced by
    a human annotator or a separately validated entailment/judge pipeline.
    """
    top_k = retrieved_chunk_ids[:k]
    gold = set(gold_chunk_ids)

    # Retrieval stage: identify which returned chunks contain gold evidence.
    binary_relevance = [int(chunk_id in gold) for chunk_id in top_k]
    retrieved_gold = set(top_k) & gold

    precision_k = sum(binary_relevance) / k
    recall_k = len(retrieved_gold) / len(gold) if gold else 0.0
    reciprocal_rank = next(
        (1.0 / rank for rank, rel in enumerate(binary_relevance, start=1) if rel),
        0.0,
    )

    # Generation stage: proportion of answer claims supported by retrieved context.
    faithfulness = (
        sum(claim_support_labels) / len(claim_support_labels)
        if claim_support_labels
        else 0.0
    )

    return {
        f"precision@{k}": precision_k,
        f"recall@{k}": recall_k,
        f"mrr@{k}": reciprocal_rank,
        "answer_faithfulness": faithfulness,
    }


metrics = evaluate_rag_example(
    retrieved_chunk_ids=["weather_12", "loyalty_03", "airport_07"],
    gold_chunk_ids=["weather_12", "airport_07", "routing_04"],
    claim_support_labels=[True, True, False],  # One generated claim is unsupported.
    k=3,
)

for metric, value in metrics.items():
    print(f"{metric}: {value:.3f}")
```

</details>

The example deliberately reports retrieval and faithfulness separately. Good retrieval does not guarantee a grounded answer, and a correct answer does not prove that the system used its evidence rather than unsupported model memory.

The ranking metrics can be selected with the following practical guide:

| Metric | Primary question | Best fit | Main blind spot |
|---|---|---|---|
| Precision@K | how focused is the top $K$? | small result pages or limited RAG context | ignores relevant items outside top $K$ |
| Recall@K | how much known evidence was recovered? | multi-evidence QA and high-coverage retrieval | needs reasonably complete relevance labels |
| MRR | how early is the first useful result? | FAQ and one-answer retrieval | ignores later relevant results |
| MAP | are all binary-relevant items ranked early? | multiple relevant documents per query | cannot represent relevance strength |
| NDCG | are highly relevant items ranked above partly relevant ones? | graded relevance and reranker evaluation | depends on consistent relevance grades |

A practical RAG report should normally include at least one coverage metric such as Recall@K, one ranking-quality metric such as MRR or NDCG, and generation-side measures for faithfulness and answer quality. Report results by query type, source, language, and answer complexity as well as overall averages. This turns evaluation from a leaderboard number into a diagnostic tool for deciding whether to improve chunking, retrieval, reranking, context construction, or generation.

### **Robustness and Generalization Evaluation** {#robustness-and-generalization-evaluation}

A model generalizes when it performs well on unseen examples drawn from the conditions it is expected to face. A model is robust when reasonable changes in those conditions do not cause disproportionate failures. These ideas go beyond reporting one score on a random test split: the central question is whether the test set represents the linguistic variation, users, domains, time periods, and failure pressures of deployment.

In NLP, the same underlying meaning can be expressed through formal prose, slang, abbreviations, spelling errors, code-switching, dialects, or another language. Data also changes over time: product names, public events, policies, and user intents evolve. A model may therefore achieve strong in-domain performance while failing on text that is still perfectly understandable to a human.

A useful robustness evaluation has four layers:

```text
clean in-domain set -> naturally shifted sets -> controlled stress tests
-> subgroup, confidence, and error analysis
```

The clean set establishes ordinary task performance. Naturally shifted sets test realistic changes such as a new domain or time period. Controlled stress tests isolate particular sensitivities such as typos or distractors. Subgroup and confidence analysis then show who is affected and whether the model knows when it is unreliable.

#### **Domain Shift** {#domain-shift}

Let the source or training distribution be $P_s(X,Y)$ and the target or deployment distribution be $P_t(X,Y)$. $X$ represents input text and $Y$ represents the desired label or output. Domain shift means these distributions are not identical:

$$
P_s(X,Y) \neq P_t(X,Y)
$$

The change can occur in different parts of the relationship:

| Shift type | Distributional change | NLP example |
|---|---|---|
| covariate shift | $P_s(X) \neq P_t(X)$ while $P(Y\mid X)$ is approximately stable | sentiment training uses polished reviews, deployment receives short social-media posts |
| label shift | $P_s(Y) \neq P_t(Y)$ while $P(X\mid Y)$ is approximately stable | urgent support requests become more common after a service outage |
| concept shift | $P_s(Y\mid X) \neq P_t(Y\mid X)$ | the meaning of an intent label changes after a company changes its refund policy |

These assumptions are idealized. Real deployments can contain several shifts at once. For example, moving a moderation model to a new platform changes language style, topic frequencies, user demographics, and possibly the operational definition of harmful content.

Common NLP domain shifts include:

- **topic shift:** politics to health, or consumer electronics to restaurant reviews;
- **source shift:** news articles to Reddit posts or customer chat;
- **temporal shift:** training on historical language and testing on newly emerging events;
- **geographic or cultural shift:** language use differs across regions and communities;
- **language and dialect shift:** standard English to a dialect, code-switching, or another language;
- **annotation-policy shift:** the target definition or labeling guideline changes.

Evaluation should preserve the domain boundary instead of randomly mixing all examples. A temporal deployment should use a future time period as the target test set. A cross-domain model should be tested on held-out domains. If examples from every domain are randomly distributed across train and test, the experiment measures interpolation within a mixture, not generalization to a new domain.

For any metric $M$ where larger is better, the generalization gap can be written as:

$$
\Delta_{gen}=M_{ID}-M_{target}
$$

$M_{ID}$ is performance on an in-distribution test set and $M_{target}$ is performance on a shifted target set. A small gap is desirable, but the target score must still be acceptable. A weak model can have a small gap simply because it performs poorly everywhere.

Report both the average target score and the worst domain or slice:

$$
M_{worst}=\min_{g\in G}M_g
$$

$G$ is the set of domains or groups and $M_g$ is the metric within group $g$. Worst-group evaluation prevents a large, easy domain from hiding failure on a smaller deployment-critical domain.

<details>
<summary>Python Measuring Domain Generalization Gaps</summary>

```python
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

# Each row stores a prediction plus the domain from which the text came.
results = pd.DataFrame({
    "domain": [
        "movie_reviews", "movie_reviews", "movie_reviews", "movie_reviews",
        "social_media", "social_media", "social_media", "social_media",
        "support_chat", "support_chat", "support_chat", "support_chat",
    ],
    "y_true": [1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0],
    "y_pred": [1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1],
})

def evaluate_slice(frame):
    """Return metrics for one domain without mixing its examples with others."""
    return pd.Series({
        "examples": len(frame),
        "accuracy": accuracy_score(frame["y_true"], frame["y_pred"]),
        "macro_f1": f1_score(
            frame["y_true"], frame["y_pred"], average="macro"
        ),
    })

# Step 1: calculate a metric table for every deployment domain.
domain_metrics = results.groupby("domain").apply(evaluate_slice)

# Step 2: treat movie reviews as the in-domain baseline.
id_f1 = domain_metrics.loc["movie_reviews", "macro_f1"]
domain_metrics["f1_gap_from_id"] = id_f1 - domain_metrics["macro_f1"]

# Step 3: inspect the worst domain, not only the pooled average.
worst_domain = domain_metrics["macro_f1"].idxmin()

print(domain_metrics)
print("\nWorst target domain:", worst_domain)
```

</details>

Domain metadata should be defined before looking at results whenever possible. Creating slices only after observing errors can still be useful for diagnosis, but it should be distinguished from pre-specified confirmatory evaluation.

#### **Adversarial Examples** {#adversarial-examples}

An adversarial example is an input deliberately modified to induce a model error while preserving the true task label or intended meaning. In text classification, an original input $x$ with label $y$ is transformed into $x'$ such that:

$$
f(x)=y,
\qquad
f(x')\neq y,
\qquad
x'\approx x
$$

$f$ is the model, and $x'\approx x$ means the perturbation should preserve meaning and remain acceptable to a human reader. This last requirement is crucial. If `The service was excellent` is changed to `The service was terrible`, the label should change; fooling the model with this sentence is not a valid label-preserving attack.

NLP attacks can operate at several levels:

| Level | Example transformation | Evaluation concern |
|---|---|---|
| character | `refund` → `refnud` | readability and realistic typo frequency |
| word | replace a word with a context-appropriate synonym | semantic and grammatical preservation |
| phrase or syntax | active to passive voice, paraphrasing | whether meaning and label remain unchanged |
| context | insert irrelevant or misleading sentences | susceptibility to distractors |
| instruction | prompt injection in tool-using or RAG systems | whether untrusted text overrides system intent |

Robustness evaluation must measure both **attack effectiveness** and **adversarial-example quality**. An attack that changes every label by producing unreadable nonsense reveals little about realistic model behavior.

Let $C$ be the set of examples the model originally classified correctly. Attack Success Rate (ASR) is commonly defined as:

$$
\operatorname{ASR}
=\frac{\sum_{i\in C}\mathbb{1}[f(x'_i)\neq y_i]}{|C|}
$$

$\mathbb{1}[\cdot]$ equals 1 when the attacked prediction is wrong. Restricting the denominator to originally correct examples prevents the attack from receiving credit for errors that already existed.

Robust accuracy evaluates the perturbed set directly:

$$
\operatorname{RobustAccuracy}
=\frac{1}{N}\sum_{i=1}^{N}\mathbb{1}[f(x'_i)=y_i]
$$

A complete report should include clean accuracy, robust accuracy, ASR, perturbation rate, semantic similarity, and human validation of label preservation and fluency.

<details>
<summary>Python Evaluating an NLP Perturbation Set</summary>

```python
import pandas as pd

# A tiny keyword model makes the evaluation mechanics visible.
# Replace this function with a real classifier in an actual experiment.
def predict_sentiment(text):
    positive_words = {"excellent", "great", "helpful", "smooth"}
    tokens = set(text.lower().replace(".", "").split())
    return 1 if tokens & positive_words else 0


examples = pd.DataFrame({
    "original": [
        "The support agent was helpful.",
        "The checkout process was smooth.",
        "The service was excellent.",
    ],
    "perturbed": [
        "The support agent was very helpful.",       # harmless insertion
        "The checkout process was seamless.",       # synonym not in keyword list
        "The service was excelent.",                 # realistic misspelling
    ],
    "label": [1, 1, 1],
    # In a real study, humans should verify that every perturbation keeps its label.
    "label_preserved": [True, True, True],
})

examples["clean_pred"] = examples["original"].map(predict_sentiment)
examples["adversarial_pred"] = examples["perturbed"].map(predict_sentiment)

clean_correct = examples["clean_pred"] == examples["label"]
robust_correct = examples["adversarial_pred"] == examples["label"]

clean_accuracy = clean_correct.mean()
robust_accuracy = robust_correct.mean()

# Count a successful attack only when the original was correct and meaning stayed.
eligible = clean_correct & examples["label_preserved"]
attack_success_rate = (
    (~robust_correct & eligible).sum() / eligible.sum()
    if eligible.sum() else 0.0
)

print(examples[["original", "perturbed", "clean_pred", "adversarial_pred"]])
print(f"\nClean accuracy: {clean_accuracy:.3f}")
print(f"Robust accuracy: {robust_accuracy:.3f}")
print(f"Attack success rate: {attack_success_rate:.3f}")
```

</details>

Adversarial tests are diagnostic stress tests, not estimates of ordinary traffic unless the attack distribution resembles deployment. They should complement naturally occurring noise sets, such as real typos or real prompt-injection attempts, rather than replace them.

#### **Out-of-Distribution Evaluation** {#out-of-distribution-evaluation}

An out-of-distribution (OOD) input lies outside the distribution or task scope represented by training data. Domain shift and OOD are related but not identical. Under domain shift, the input may still belong to the known task and label space. In semantic OOD, none of the model's labels may be appropriate.

For an intent classifier trained on `billing`, `technical support`, and `account cancellation`:

- `pls close my acct` is stylistically shifted but still belongs to `account cancellation`;
- `Will it rain tomorrow?` is semantic OOD because no supported intent fits;
- a new `data deletion request` may be near-OOD: it is related to account management but requires a new operational route.

Evaluation should use at least three sets:

1. **ID test set:** normal unseen examples from supported intents;
2. **near-OOD set:** related topics or new intents that are easy to confuse with known labels;
3. **far-OOD set:** clearly unrelated inputs.

Maximum softmax probability is a simple OOD score:

$$
s_{MSP}(x)=\max_k P(Y=k\mid x)
$$

$k$ indexes known classes. Lower $s_{MSP}$ is treated as more OOD-like. Predictive entropy provides another uncertainty score:

$$
H(Y\mid x)=-\sum_{k=1}^{K}p_k\log p_k
$$

$p_k=P(Y=k\mid x)$ and $K$ is the number of known classes. Entropy is high when probability mass is spread across classes. However, neural networks can remain highly confident on unfamiliar text, so confidence is only a baseline detector.

OOD detection is a ranking problem over ID and OOD scores. Common metrics include:

- **AUROC:** probability that a randomly chosen OOD example receives a more OOD-like score than a randomly chosen ID example;
- **AUPR:** precision-recall performance, which is sensitive to the chosen positive class and useful under imbalance;
- **FPR@95TPR:** proportion of ID examples falsely rejected when 95% of OOD examples are detected, under the convention that OOD is positive.

Always state which class is positive and which direction indicates OOD. Otherwise AUROC or AUPR results can be misinterpreted.

<details>
<summary>Python OOD Detection from Model Confidence</summary>

```python
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

# Maximum predicted probabilities from a known-intent classifier.
id_confidence = np.array([0.96, 0.88, 0.79, 0.91, 0.72, 0.84])
ood_confidence = np.array([0.62, 0.31, 0.55, 0.81, 0.44, 0.28])

# OOD is the positive class. Convert confidence into an OOD score:
# lower model confidence -> higher OOD score.
y_true = np.concatenate([
    np.zeros(len(id_confidence)),
    np.ones(len(ood_confidence)),
])
ood_score = 1.0 - np.concatenate([id_confidence, ood_confidence])

auroc = roc_auc_score(y_true, ood_score)
aupr = average_precision_score(y_true, ood_score)

# Find the lowest FPR among operating points detecting at least 95% of OOD cases.
fpr, tpr, thresholds = roc_curve(y_true, ood_score)
eligible = np.where(tpr >= 0.95)[0]
fpr_at_95_tpr = fpr[eligible].min() if len(eligible) else float("nan")

print(f"OOD AUROC: {auroc:.3f}")
print(f"OOD AUPR: {aupr:.3f}")
print(f"FPR@95TPR: {fpr_at_95_tpr:.3f}")
```

</details>

For deployment, OOD detection is usually connected to selective prediction: low-confidence examples are rejected, routed to a fallback model, or sent for human review. Report the **risk-coverage curve**, where coverage is the fraction of examples answered and risk is the error rate among those answered. A useful abstention policy should reduce risk smoothly as coverage decreases.

#### **Bias and Fairness Evaluation** {#bias-and-fairness-evaluation}

Bias and fairness evaluation asks whether model quality, errors, or treatment differ systematically across socially or operationally important groups. There is no universal fairness metric independent of context. The appropriate definition depends on the task, harm, decision process, and whether the group attribute is ethically and legally appropriate to collect.

In NLP, bias can enter through training corpora, annotation guidelines, label imbalance, identity terms, dialect differences, retrieval coverage, and the evaluator itself. Examples include:

- a toxicity classifier producing more false positives for text containing identity terms;
- an intent classifier misunderstanding dialectal or non-native language;
- a coreference model associating occupations with gender stereotypes;
- a retrieval system exposing lower-quality sources for some languages;
- a generative model producing stereotyped descriptions or unequal refusals.

For classification, group-specific true-positive and false-positive rates are:

$$
TPR_g=\frac{TP_g}{TP_g+FN_g},
\qquad
FPR_g=\frac{FP_g}{FP_g+TN_g}
$$

$g$ identifies a group. $TP_g$, $FN_g$, $FP_g$, and $TN_g$ are confusion-matrix counts within that group. Equal opportunity focuses on similar TPR across groups; equalized odds considers both TPR and FPR. The gap for a metric $M$ can be summarized as:

$$
\operatorname{Gap}(M)=\max_{g\in G}M_g-\min_{g\in G}M_g
$$

A small gap is not sufficient by itself. All groups could have equally poor performance. Report each group score, the gap, sample sizes, and uncertainty.

Counterfactual evaluation creates paired texts that differ only in a sensitive identity reference while preserving the task label:

```text
"He is a skilled engineer." <-> "She is a skilled engineer."
```

The counterfactual flip rate is:

$$
\operatorname{CFR}
=\frac{1}{N}\sum_{i=1}^{N}\mathbb{1}[f(x_i)\neq f(x'_i)]
$$

$x_i$ and $x'_i$ form a controlled pair. A flip can reveal sensitivity, but interpreting it requires care: some identity changes can legitimately alter meaning in particular contexts, and simplistic word replacement can create unnatural text.

<details>
<summary>Python Group Metrics and Counterfactual Flip Rate</summary>

```python
import pandas as pd
from sklearn.metrics import confusion_matrix

results = pd.DataFrame({
    "group": ["A"] * 6 + ["B"] * 6,
    "y_true": [1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0],
    "y_pred": [1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1],
})

def group_error_rates(frame):
    """Compute rates separately so pooled data cannot hide group differences."""
    tn, fp, fn, tp = confusion_matrix(
        frame["y_true"], frame["y_pred"], labels=[0, 1]
    ).ravel()
    return pd.Series({
        "n": len(frame),
        "tpr": tp / (tp + fn) if tp + fn else float("nan"),
        "fpr": fp / (fp + tn) if fp + tn else float("nan"),
    })


group_metrics = results.groupby("group").apply(group_error_rates)
tpr_gap = group_metrics["tpr"].max() - group_metrics["tpr"].min()
fpr_gap = group_metrics["fpr"].max() - group_metrics["fpr"].min()

# Paired predictions from identity-swapped sentences.
original_predictions = [0, 1, 0, 0, 1]
counterfactual_predictions = [0, 0, 0, 1, 1]
flip_rate = sum(
    left != right
    for left, right in zip(original_predictions, counterfactual_predictions)
) / len(original_predictions)

print(group_metrics)
print(f"\nTPR gap: {tpr_gap:.3f}")
print(f"FPR gap: {fpr_gap:.3f}")
print(f"Counterfactual flip rate: {flip_rate:.3f}")
```

</details>

Fairness evaluation should include intersectional slices where data permits, such as language plus region or identity plus dialect, because aggregate group labels can hide concentrated failure. Small groups require confidence intervals or careful qualitative review. Sensitive attributes should never be inferred casually from names or text when reliable and ethically collected metadata is unavailable.

For generative systems, classification rate gaps are not enough. Use structured prompts and human review to examine stereotypes, representational harms, quality of service, refusal behavior, and toxicity. Automated judges used for fairness must themselves be validated for group-dependent bias.

#### **Calibration and Confidence** {#calibration-and-confidence}

Confidence answers how strongly a model favors its prediction. Calibration asks whether that confidence corresponds to empirical correctness. A perfectly calibrated classifier satisfies:

$$
P(\hat{Y}=Y\mid \hat{P}=p)=p
$$

$\hat{Y}$ is the predicted label, $Y$ is the true label, and $\hat{P}$ is predicted confidence. If a model makes 100 predictions at confidence 0.8, approximately 80 should be correct. Calibration does not require every 0.8-confidence prediction to be correct; it requires correctness frequency to match confidence across many comparable predictions.

![Confidence histograms and reliability diagrams](assets/confidence-reliability-diagram.png){fig-alt="Confidence histograms and reliability diagrams comparing a better calibrated LeNet with an overconfident ResNet." width=68%}

*Source: [On Calibration of Modern Neural Networks](https://proceedings.mlr.press/v70/guo17a.html), Figure 1.*

The bottom plots are reliability diagrams. Confidence bins appear on the horizontal axis and empirical accuracy on the vertical axis. A calibrated model follows the diagonal. Bars below the diagonal indicate overconfidence: predicted confidence is higher than observed accuracy. Bars above it indicate underconfidence.

Expected Calibration Error (ECE) summarizes the average bin-wise gap:

$$
\operatorname{ECE}
=\sum_{m=1}^{M}\frac{|B_m|}{N}
\left|\operatorname{acc}(B_m)-\operatorname{conf}(B_m)\right|
$$

$M$ is the number of confidence bins, $B_m$ contains examples assigned to bin $m$, $|B_m|/N$ is that bin's proportion of all $N$ examples, $\operatorname{acc}(B_m)$ is empirical accuracy, and $\operatorname{conf}(B_m)$ is mean predicted confidence. ECE is zero for perfect empirical calibration under the chosen binning.

ECE depends on the number and boundaries of bins and can hide class-specific or subgroup-specific miscalibration. It should be paired with a reliability diagram and proper scoring rules such as Negative Log-Likelihood or the Brier score. For binary classification, the Brier score is:

$$
\operatorname{Brier}
=\frac{1}{N}\sum_{i=1}^{N}(p_i-y_i)^2
$$

$p_i$ is predicted positive-class probability and $y_i\in\{0,1\}$ is the true label. Unlike accuracy, it penalizes the distance between probability and outcome. Confident mistakes receive a large penalty.

<details>
<summary>Python ECE and Reliability Diagram</summary>

```python
import numpy as np
import matplotlib.pyplot as plt

def calibration_table(y_true, y_pred, confidence, n_bins=10):
    """Return bin statistics used by both ECE and a reliability diagram."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    confidence = np.asarray(confidence)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []

    for lower, upper in zip(edges[:-1], edges[1:]):
        # Include confidence 1.0 in the final bin.
        in_bin = (confidence > lower) & (confidence <= upper)
        if lower == 0.0:
            in_bin = (confidence >= lower) & (confidence <= upper)
        if not in_bin.any():
            continue

        rows.append({
            "midpoint": (lower + upper) / 2,
            "count": int(in_bin.sum()),
            "accuracy": float((y_pred[in_bin] == y_true[in_bin]).mean()),
            "confidence": float(confidence[in_bin].mean()),
        })
    return rows


y_true = np.array([1, 0, 1, 1, 0, 0, 1, 0, 1, 0])
y_pred = np.array([1, 1, 1, 1, 0, 1, 0, 0, 1, 0])
confidence = np.array([0.95, 0.90, 0.82, 0.76, 0.72, 0.68, 0.65, 0.61, 0.58, 0.55])

rows = calibration_table(y_true, y_pred, confidence, n_bins=5)
n = len(y_true)
ece = sum(
    row["count"] / n * abs(row["accuracy"] - row["confidence"])
    for row in rows
)

# Plot empirical accuracy against mean confidence for each occupied bin.
plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], "--", color="gray", label="perfect calibration")
plt.plot(
    [row["confidence"] for row in rows],
    [row["accuracy"] for row in rows],
    marker="o",
    label="model",
)
plt.xlabel("Mean confidence")
plt.ylabel("Empirical accuracy")
plt.title(f"Reliability diagram (ECE={ece:.3f})")
plt.legend()
plt.grid(alpha=0.25)
plt.show()
```

</details>

Temperature scaling is a common post-hoc calibration method. Given logits $z_1,\ldots,z_K$, it computes:

$$
P_T(Y=k\mid x)
=\frac{\exp(z_k/T)}{\sum_{j=1}^{K}\exp(z_j/T)}
$$

$T>0$ is learned on a held-out calibration set by minimizing NLL. When $T>1$, the probability distribution becomes softer and usually less confident. When $0<T<1$, it becomes sharper. Dividing all logits by one positive scalar does not change their ordering, so temperature scaling normally changes confidence without changing predicted labels or accuracy.

<details>
<summary>PyTorch Temperature Scaling on Validation Logits</summary>

```python
import torch
import torch.nn.functional as F

# Validation logits and labels must be separate from model-training data.
validation_logits = torch.tensor([
    [3.2, 0.4],
    [2.1, 1.8],
    [0.2, 2.7],
    [1.9, 2.2],
], dtype=torch.float32)
validation_labels = torch.tensor([0, 1, 1, 0])

# Optimize log(T), then exponentiate, so temperature always remains positive.
log_temperature = torch.nn.Parameter(torch.zeros(()))
optimizer = torch.optim.LBFGS([log_temperature], lr=0.1, max_iter=50)

def closure():
    optimizer.zero_grad()
    temperature = log_temperature.exp()
    loss = F.cross_entropy(validation_logits / temperature, validation_labels)
    loss.backward()
    return loss

optimizer.step(closure)
temperature = log_temperature.exp().detach()

uncalibrated = validation_logits.softmax(dim=-1)
calibrated = (validation_logits / temperature).softmax(dim=-1)

print(f"Learned temperature: {temperature.item():.3f}")
print("Confidence before:", uncalibrated.max(dim=-1).values)
print("Confidence after: ", calibrated.max(dim=-1).values)
```

</details>

Calibration must be rechecked under domain shift. A temperature fitted on movie reviews may not calibrate predictions on social media or support chat. Also distinguish sequence probability from factual confidence in generated text: a language model can assign high token probability to a fluent but false statement. Generation confidence may require claim-level verification, retrieval support, self-consistency, or calibrated external evaluators.

The five evaluation areas can be summarized as follows:

| Evaluation area | Primary question | Essential measurements |
|---|---|---|
| domain shift | does performance transfer to realistic target domains? | target score, generalization gap, worst-domain score |
| adversarial robustness | does a label-preserving perturbation cause failure? | clean and robust scores, ASR, semantic and human validity |
| OOD evaluation | can the system identify unsupported or unfamiliar inputs? | near/far-OOD sets, AUROC, AUPR, FPR@95TPR, risk-coverage |
| bias and fairness | are errors or treatment uneven across important groups? | group metrics, worst-group result, gaps, counterfactual tests |
| calibration | does reported confidence match empirical correctness? | reliability diagram, ECE, NLL or Brier score |

The practical goal is not to prove that a model is universally robust; no finite benchmark can do that. The goal is to make deployment assumptions explicit, test realistic and controlled failures, quantify uncertainty, and expose where the model should abstain or require human oversight.

### **Error Analysis** {#error-analysis}

An evaluation metric tells us **how much** a model is wrong, but rarely tells us **why** it is wrong or what should be changed next. Error analysis turns predictions into hypotheses about the data, task definition, model, threshold, or deployment pipeline.

For example, a sentiment classifier with macro F1 of 0.82 may fail mainly on negation, sarcastic comments, mixed sentiment, or domain-specific vocabulary. These failure modes suggest different actions. Negation errors may motivate behavioral tests or better training examples; domain vocabulary may require target-domain data; mixed sentiment may expose an overly simple single-label formulation.

A useful error-analysis cycle is:

```text
locate errors -> group recurring patterns -> inspect representative examples
-> form a hypothesis -> design a targeted test -> change one component -> re-evaluate
```

The goal is not to explain every individual mistake. It is to find **systematic, actionable patterns** that affect meaningful parts of the task.

#### **Qualitative Error Inspection** {#qualitative-error-inspection}

Qualitative inspection means reading the original text, gold label, model output, confidence, and relevant metadata. It is where linguistic details become visible. A confusion matrix may show that `neutral` is often predicted as `positive`, but only reading examples reveals whether the cause is politeness, implicit sentiment, annotation ambiguity, or a missing context turn.

Randomly reading a few mistakes is better than nothing, but it can miss important categories. A stronger sampling plan includes:

- a random sample of all errors, which estimates common patterns;
- high-confidence errors, which reveal confidently learned shortcuts or annotation problems;
- near-threshold cases, which reveal decision-boundary ambiguity;
- errors from each important class, domain, language, and user group;
- disagreements between models or evaluators;
- newly observed production failures and safety-critical cases.

High-confidence errors deserve special attention. A low-confidence mistake may be genuinely ambiguous, while a 0.99-confidence mistake can indicate leakage, a spurious keyword, calibration failure, or a systematic labeling mismatch.

Every inspected example should distinguish at least four possible causes:

| Cause family | Diagnostic question | Example |
|---|---|---|
| data or annotation | is the gold label correct and the input complete? | sarcasm labeled inconsistently |
| representation or preprocessing | was useful information removed or split badly? | negation dropped during cleaning |
| model behavior | did the model rely on a shortcut or miss a linguistic relation? | predicts positive whenever `great` appears, even under negation |
| task or product design | is one label sufficient for the real case? | mixed sentiment forced into one class |

<details>
<summary>Python Sampling High-Value Errors for Inspection</summary>

```python
import pandas as pd

predictions = pd.DataFrame({
    "text": [
        "Great, another two-hour delay.",
        "The food was good but the service was awful.",
        "I cannot recommend this hotel.",
        "Thanks for resolving the issue so quickly.",
        "It was fine, I suppose.",
        "Absolutely fantastic support!",
    ],
    "gold": ["negative", "mixed", "negative", "positive", "neutral", "positive"],
    "predicted": ["positive", "negative", "positive", "positive", "positive", "positive"],
    "confidence": [0.97, 0.71, 0.93, 0.88, 0.56, 0.99],
    "domain": ["social", "review", "review", "support", "review", "support"],
})

# Step 1: isolate mistakes without discarding their original text or metadata.
errors = predictions[predictions["gold"] != predictions["predicted"]].copy()

# Step 2: select complementary samples rather than only the easiest examples.
high_confidence = errors.nlargest(2, "confidence")
near_boundary = errors.iloc[(errors["confidence"] - 0.5).abs().argsort()[:2]]
per_domain = errors.groupby("domain", group_keys=False).head(1)

# Step 3: combine and de-duplicate examples for manual review.
review_sample = pd.concat(
    [high_confidence, near_boundary, per_domain]
).drop_duplicates(subset="text")

print(review_sample[["text", "gold", "predicted", "confidence", "domain"]])
```

</details>

Inspection should be blinded when feasible: reviewers should first judge whether the gold label and model prediction are defensible before seeing model identity. Otherwise expectations about a model can influence the diagnosis.

#### **Slice-Based Evaluation** {#slice-based-evaluation}

A slice is a meaningful subset of the evaluation data. Overall performance is a weighted average across examples, so a large, easy slice can hide severe failure on a smaller slice.

For metric $M$ and slice $S_g$, define:

$$
M_g=M\left(\{(x_i,y_i,\hat{y}_i):i\in S_g\}\right)
$$

$x_i$ is input text, $y_i$ is the gold output, and $\hat{y}_i$ is the prediction. The difference from overall performance is:

$$
\Delta_g=M_g-M_{overall}
$$

A strongly negative $\Delta_g$ identifies an underperforming slice when larger values of $M$ are better. The slice may be based on metadata or text properties:

- domain, source, language, region, or time period;
- input length or conversation depth;
- class label or entity type;
- presence of negation, numbers, named entities, quotations, or code-switching;
- frequent versus rare vocabulary;
- retrieval difficulty or number of supporting passages;
- human-defined linguistic capabilities such as coreference or temporal reasoning.

Slice definitions should be interpretable and connected to an action. A cluster described only as `embedding cluster 17` is less useful than `long support messages containing several issue types`. Automatically discovered slices are valuable for exploration, but humans should verify their coherence.

<details>
<summary>Python NLP Slice-Based Evaluation</summary>

```python
import pandas as pd
from sklearn.metrics import f1_score

results = pd.DataFrame({
    "text": [
        "Excellent service",
        "I do not like this product",
        "Fine",
        "The device worked at first but failed after the update",
        "I wouldn't say the experience was good",
        "Fast delivery and helpful support",
    ],
    "gold": [1, 0, 1, 0, 0, 1],
    "predicted": [1, 1, 1, 0, 1, 1],
    "domain": ["review", "review", "social", "support", "social", "support"],
})

# Step 1: create interpretable text-derived slices.
results["has_negation"] = results["text"].str.contains(
    r"\b(no|not|never|wouldn't|isn't|can't|cannot)\b",
    case=False,
    regex=True,
)
results["length_slice"] = pd.cut(
    results["text"].str.split().str.len(),
    bins=[0, 4, 9, float("inf")],
    labels=["short", "medium", "long"],
)

def slice_f1(frame):
    return f1_score(frame["gold"], frame["predicted"], average="macro")

# Step 2: compare pre-defined operational and linguistic slices.
overall_f1 = slice_f1(results)
domain_scores = results.groupby("domain").apply(slice_f1)
negation_scores = results.groupby("has_negation").apply(slice_f1)
length_scores = results.groupby("length_slice", observed=True).apply(slice_f1)

print(f"Overall macro F1: {overall_f1:.3f}")
print("\nBy domain:\n", domain_scores)
print("\nBy negation:\n", negation_scores)
print("\nBy length:\n", length_scores)
```

</details>

Always report slice size and, for small slices, uncertainty intervals. Testing many slices also increases the chance of finding an apparently bad slice by chance. Treat discovered problems as hypotheses and confirm them on new or held-out examples.

#### **False Positives and False Negatives** {#false-positives-and-false-negatives}

For a target class, a false positive (FP) occurs when the model predicts the class but the gold label is different. A false negative (FN) occurs when the gold label belongs to the class but the model misses it.

The distinction is operational, not merely mathematical. In toxicity detection:

- a false positive may incorrectly hide harmless speech or unfairly penalize a user;
- a false negative may leave harmful content visible.

In medical information extraction:

- a false positive may add a condition not supported by the note;
- a false negative may omit a clinically important condition.

The preferred balance depends on downstream consequences. A cost-sensitive view assigns costs $C_{FP}$ and $C_{FN}$:

$$
\operatorname{TotalCost}=C_{FP}\cdot FP+C_{FN}\cdot FN
$$

$FP$ and $FN$ are error counts. The costs do not have to be monetary; they may represent user harm, manual-review effort, missed incidents, or another decision-relevant quantity.

Error inspection should also use scores. High-confidence false positives and false negatives often reveal different model behavior from borderline cases.

<details>
<summary>Python Extracting Ranked False Positives and False Negatives</summary>

```python
import pandas as pd

results = pd.DataFrame({
    "text": [
        "I hate waiting in traffic",
        "You are completely useless",
        "That idea is not stupid",
        "Go away, nobody wants you here",
        "This movie was painfully slow",
    ],
    "gold_toxic": [0, 1, 0, 1, 0],
    "toxic_probability": [0.82, 0.91, 0.77, 0.43, 0.35],
})

threshold = 0.5
results["predicted_toxic"] = (
    results["toxic_probability"] >= threshold
).astype(int)

# Step 1: identify error direction.
false_positives = results[
    (results["gold_toxic"] == 0) & (results["predicted_toxic"] == 1)
].sort_values("toxic_probability", ascending=False)

false_negatives = results[
    (results["gold_toxic"] == 1) & (results["predicted_toxic"] == 0)
].sort_values("toxic_probability", ascending=True)

# Step 2: review confident errors first, while retaining borderline errors too.
print("False positives:\n", false_positives[["text", "toxic_probability"]])
print("\nFalse negatives:\n", false_negatives[["text", "toxic_probability"]])
```

</details>

Changing the threshold redistributes FP and FN but does not repair poor ranking or representation. If positive and negative score distributions overlap heavily, threshold tuning alone cannot satisfy both precision and recall requirements.

#### **Confusion Patterns** {#confusion-patterns}

In multi-class tasks, an error is not just `wrong`; it is a transition from one gold label to another predicted label. Repeated transitions reveal semantic boundaries the model has not learned.

For an intent classifier, common patterns might include:

```text
gold: refund_status       -> predicted: request_refund
gold: cancel_subscription -> predicted: close_account
gold: payment_failed      -> predicted: card_declined
```

These pairs may expose overlapping label definitions, insufficient dialogue context, or training examples that do not clearly separate neighboring intents. A normalized confusion matrix helps compare classes of different sizes, but the original counts must also be retained.

Sequence tasks need richer confusion analysis. In NER, distinguish:

- correct span but wrong entity type;
- correct type but left or right boundary error;
- completely missed entity;
- spurious entity;
- invalid BIO transition.

Generation tasks can use error transitions between categories such as `supported claim -> unsupported claim`, `complete answer -> omitted constraint`, or `correct citation -> wrong source attribution`.

<details>
<summary>Python Ranking Multi-Class Confusion Pairs</summary>

```python
from collections import Counter

gold = [
    "refund_status", "refund_status", "request_refund",
    "cancel_subscription", "close_account", "payment_failed",
    "payment_failed", "card_declined",
]
predicted = [
    "request_refund", "refund_status", "request_refund",
    "close_account", "close_account", "card_declined",
    "payment_failed", "payment_failed",
]

# Count directional confusions; A -> B is different from B -> A.
confusions = Counter(
    (true_label, predicted_label)
    for true_label, predicted_label in zip(gold, predicted)
    if true_label != predicted_label
)

for (true_label, predicted_label), count in confusions.most_common():
    print(f"{true_label:22s} -> {predicted_label:22s}: {count}")
```

</details>

A confusion pair becomes actionable only after examples are read. The same transition can come from label ambiguity in one case and missing linguistic understanding in another.

#### **Building an Error Analysis Table** {#building-an-error-analysis-table}

An error analysis table is the bridge between unstructured inspection and reproducible diagnosis. Each row should preserve enough information for another reviewer to understand the example and reproduce the prediction.

A useful schema includes:

| Field | Purpose |
|---|---|
| example ID and text | trace the original input without losing context |
| gold and prediction | record the observed mismatch |
| score or confidence | separate borderline from confident errors |
| domain and slice metadata | locate concentrated failure |
| primary error category | assign one dominant diagnosis for counting |
| secondary tags | preserve overlapping phenomena such as negation plus sarcasm |
| annotation status | flag incorrect or ambiguous gold labels |
| severity and impact | prioritize operationally important failures |
| proposed action | connect diagnosis to data, model, threshold, or product changes |
| reviewer and notes | make judgments auditable |

Use a controlled taxonomy rather than inventing a new phrase for every row. The taxonomy can evolve, but changes should be versioned. A practical top-level taxonomy is:

```text
annotation | missing context | preprocessing | lexical | negation | coreference
temporal | numerical | domain knowledge | ambiguity | threshold | safety | other
```

<details>
<summary>Python Creating an Auditable Error Analysis Table</summary>

```python
import pandas as pd

errors = pd.DataFrame({
    "example_id": ["ex_001", "ex_002", "ex_003"],
    "text": [
        "Great, another delay.",
        "I cannot recommend it.",
        "The service was good, but the product failed immediately.",
    ],
    "gold": ["negative", "negative", "mixed"],
    "predicted": ["positive", "positive", "negative"],
    "confidence": [0.97, 0.93, 0.72],
    "domain": ["social", "review", "review"],
})

# Add review fields before annotation so every reviewer uses the same schema.
for column in [
    "primary_error", "secondary_tags", "annotation_status",
    "severity", "proposed_action", "reviewer_notes",
]:
    errors[column] = ""

# Example diagnoses after manual inspection.
errors.loc[0, ["primary_error", "severity", "proposed_action"]] = [
    "sarcasm", "medium", "add sarcasm behavioral test",
]
errors.loc[1, ["primary_error", "severity", "proposed_action"]] = [
    "negation", "high", "expand negation training and invariance tests",
]
errors.loc[2, ["primary_error", "annotation_status", "proposed_action"]] = [
    "task_formulation", "valid", "support mixed or aspect-level sentiment",
]

# Save as UTF-8 so the table can be reviewed by humans and aggregated later.
errors.to_csv("error_analysis.csv", index=False, encoding="utf-8")
print(errors)
```

</details>

After enough examples are labeled, aggregate by error category, severity, class, and domain. Prioritize categories that are frequent, harmful, and realistically fixable. Then create regression tests from representative failures so that later model changes do not silently reintroduce them.

### **Choosing the Right Evaluation Metric** {#choosing-the-right-evaluation-metric}

There is no universally best NLP metric. A metric is appropriate when it matches the task output, data distribution, decision cost, and claim being made. Metric selection should happen before final model comparison; otherwise it is easy to choose whichever score makes a preferred model look strongest.

A good evaluation normally defines:

- one **primary metric** tied to the main objective;
- secondary metrics that expose important trade-offs;
- guardrail metrics that must not regress, such as safety or worst-group performance;
- slice metrics for known risks;
- qualitative or human evaluation when automatic metrics are incomplete.

#### **Task Type** {#task-type}

The output structure determines the first set of candidate metrics:

| NLP task | Output structure | Useful primary metrics | Important companion checks |
|---|---|---|---|
| balanced single-label classification | one class | accuracy, macro F1 | per-class metrics and confusion matrix |
| imbalanced detection | positive or negative | PR-AUC, target-class F1 | precision/recall at deployment threshold |
| multi-label classification | several labels | micro/macro F1, average precision | label coverage and per-label results |
| NER or slot filling | labeled spans | exact entity/span F1 | boundary and type errors, token sanity check |
| language modeling | token probabilities | mean NLL, perplexity | tokenizer, domain slices, generation quality |
| translation | generated sequence | corpus BLEU, learned metrics | human adequacy and terminology checks |
| summarization | generated sequence | ROUGE plus semantic metric | factuality, coverage, human usefulness |
| retrieval | ranked candidates | Recall@K, MRR, MAP, NDCG | latency, corpus coverage, query slices |
| RAG | retrieved context plus answer | Recall@K plus faithfulness/correctness | citation support and end-to-end task success |
| dialogue or open generation | open-ended response | validated judge or human preference | safety, factuality, instruction following |

The metric must use the same unit as the task objective. Token accuracy is not sufficient for entity extraction, and answer similarity is not sufficient for source-grounded factuality.

#### **Class Imbalance** {#class-imbalance}

When one class dominates, accuracy and micro averages can look strong even if minority classes fail. Suppose 99% of messages are safe. A classifier that always predicts `safe` reaches 99% accuracy but has zero recall for harmful messages.

Metric choice should reflect the minority-class role:

- use class-specific precision when false alarms are costly;
- use class-specific recall when missed cases are costly;
- use F1 when both matter and one operating threshold is needed;
- use macro F1 when each class deserves equal reporting weight;
- use PR-AUC to assess ranking quality for a rare positive class across thresholds;
- include support counts and per-class metrics so averages remain interpretable.

Resampling the evaluation set can change apparent precision because precision depends on prevalence. If a balanced test set is created artificially, its precision may not represent deployment. Keep a naturally distributed test set or adjust interpretation with deployment prevalence.

#### **Business or Research Objective** {#business-or-research-objective}

A product objective describes the consequence of predictions. A research objective describes the capability or hypothesis under study. Both should be translated into an evaluation contract.

For a binary system with false-positive cost $C_{FP}$ and false-negative cost $C_{FN}$, average decision cost is:

$$
\operatorname{AverageCost}
=\frac{C_{FP}\cdot FP+C_{FN}\cdot FN}{N}
$$

$N$ is the number of evaluated examples. A safety triage system may assign a larger cost to false negatives. An automatic account-blocking system may assign a large cost to false positives. The same classifier scores can therefore justify different thresholds in different products.

Operational constraints may be expressed directly:

```text
maximize recall subject to precision >= 0.95
maximize answer quality subject to latency <= 2 seconds
minimize review volume subject to missed-risk rate <= 1%
```

Research comparisons need additional discipline: a fixed benchmark, unchanged preprocessing, repeated seeds where training is stochastic, uncertainty intervals, and paired significance tests. A 0.2-point improvement is not persuasive if run-to-run variation is larger.

#### **Metric Trade-Offs** {#metric-trade-offs}

Many metrics are connected by a controllable operating choice:

- classification threshold trades precision against recall;
- retrieval cutoff $K$ trades context focus against evidence coverage;
- decoding strategy trades diversity against determinism and likelihood;
- abstention threshold trades coverage against risk;
- generation length can improve recall-oriented overlap while increasing unsupported content.

Therefore, compare models at a meaningful operating point, not only at each model's most flattering threshold. If production requires precision of at least 0.95, compare recall after both models are constrained to that precision.

<details>
<summary>Python Selecting a Threshold under a Precision Constraint</summary>

```python
import numpy as np
from sklearn.metrics import precision_recall_curve

y_true = np.array([1, 0, 1, 0, 1, 0, 0, 1, 0, 1])
y_score = np.array([0.98, 0.90, 0.82, 0.70, 0.66, 0.51, 0.42, 0.39, 0.20, 0.12])

precision, recall, thresholds = precision_recall_curve(y_true, y_score)

# precision and recall contain one extra endpoint without a threshold.
candidates = [
    (threshold, p, r)
    for threshold, p, r in zip(thresholds, precision[:-1], recall[:-1])
    if p >= 0.80
]

# Among thresholds satisfying the precision requirement, retain the most recall.
best_threshold, best_precision, best_recall = max(
    candidates,
    key=lambda row: row[2],
)

print(f"Threshold: {best_threshold:.3f}")
print(f"Precision: {best_precision:.3f}")
print(f"Recall: {best_recall:.3f}")
```

</details>

Threshold selection must use validation data. Measuring many thresholds on the test set and reporting the best one turns the test set into a tuning set.

#### **When Metrics Disagree** {#when-metrics-disagree}

Metric disagreement is usually information, not an inconvenience. It means the metrics reward different behavior.

Examples include:

- accuracy improves while macro F1 falls because the majority class improved and a minority class regressed;
- BLEU falls while BERTScore rises because the candidate paraphrases instead of copying reference n-grams;
- ROUGE recall rises while human factuality falls because a longer summary covers more reference words but hallucinates details;
- Recall@K rises while Precision@K falls because more passages were retrieved with more distractors;
- average performance rises while worst-group performance falls because gains are concentrated in an easy group.

Resolve disagreement by returning to the evaluation contract:

1. identify what each metric rewards;
2. inspect examples where the metrics disagree most;
3. check aggregation, threshold, tokenizer, reference, and slice definitions;
4. determine which failure has greater task impact;
5. report the trade-off instead of hiding the unfavorable metric.

Small metric differences also require uncertainty analysis. A paired bootstrap resamples evaluation examples and recomputes the difference while preserving the fact that both models were tested on the same inputs.

<details>
<summary>Python Paired Bootstrap for a Metric Difference</summary>

```python
import numpy as np
from sklearn.metrics import accuracy_score

y_true = np.array([1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0])
pred_a = np.array([1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0])
pred_b = np.array([1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0])

rng = np.random.default_rng(42)
differences = []
n = len(y_true)

# Resample indices once per iteration and apply the same sample to both models.
for _ in range(5000):
    indices = rng.integers(0, n, size=n)
    score_a = accuracy_score(y_true[indices], pred_a[indices])
    score_b = accuracy_score(y_true[indices], pred_b[indices])
    differences.append(score_a - score_b)

observed = accuracy_score(y_true, pred_a) - accuracy_score(y_true, pred_b)
lower, upper = np.percentile(differences, [2.5, 97.5])

print(f"Observed A-B difference: {observed:.3f}")
print(f"95% bootstrap interval: [{lower:.3f}, {upper:.3f}]")
```

</details>

If the interval includes zero, the data do not clearly establish which model is better under that metric. This is not proof of equality; it means the experiment lacks strong evidence for a directional difference.

### **Practical Evaluation Workflow** {#practical-evaluation-workflow}

A practical NLP evaluation should be designed before final training and should produce reusable artifacts, not only one number. The following workflow connects the concepts developed throughout this chapter.

**Define the task and decision.** Specify the input, output, target population, deployment environment, and what happens after a prediction. Clarify whether the unit is a token, span, document, query, dialogue turn, or complete conversation.

**Write an evaluation contract.** Record the primary metric, secondary metrics, guardrails, slices, operating constraints, and acceptance criteria. For example:

```text
Primary: macro F1 >= 0.85
Constraint: urgent-intent recall >= 0.95
Guardrail: worst-language F1 must not drop by more than 0.02
Calibration: ECE <= 0.05 on the deployment-like validation set
```

**Build leakage-resistant datasets.** Separate train, validation, and test sets by the unit that could leak, such as user, document, conversation, product, or time. Deduplicate and inspect label quality. Keep a deployment-like test set and additional shift or stress sets.

**Establish simple baselines.** Compare against majority-class, lexical, TF-IDF, BM25, or other task-appropriate baselines. A complex model should demonstrate value beyond a simple system under the same evaluation protocol.

**Tune only on validation data.** Use validation data for model selection, thresholds, decoding settings, temperature scaling, prompt changes, and retrieval cutoff. Preserve the test set for the final estimate.

**Evaluate aggregate and slice performance.** Report the primary metric, per-class or per-task details, domain and demographic slices, robustness sets, calibration, latency, and uncertainty. Include counts so small slices are not mistaken for stable estimates.

**Run behavioral tests.** The [CheckList methodology](https://www.microsoft.com/en-us/research/publication/beyond-accuracy-behavioral-testing-of-nlp-models-with-checklist/) organizes NLP tests into:

- **Minimum Functionality Tests (MFT):** simple examples that test one capability, such as basic negation;
- **Invariance Tests (INV):** label-preserving changes, such as harmless name replacement or typos, should not alter predictions;
- **Directional Expectation Tests (DIR):** a controlled change should move prediction in an expected direction, such as adding clearly positive language increasing positive sentiment.

These tests complement held-out examples because they target behaviors that may be rare in a random test sample.

**Perform structured error analysis.** Inspect random, high-confidence, slice-specific, and high-severity errors. Build an error table, quantify recurring categories, and convert representative failures into regression tests.

**Compare with paired uncertainty.** Use bootstrap intervals, repeated seeds, or an appropriate paired test. Inspect disagreements rather than relying only on mean scores.

**Document and version the evaluation.** Record dataset version, label policy, preprocessing, tokenizer, model version, prompts, judge model, thresholds, decoding settings, library versions, and random seeds. For learned or LLM-based metrics, the evaluator is part of the measurement instrument and must be versioned.

**Monitor after deployment.** Track input drift, output distributions, confidence, abstention, latency, user feedback, and delayed ground-truth performance. Production failures should enter a reviewed test set without contaminating the historical test set used for earlier claims.

<details>
<summary>Python Building a Reproducible Evaluation Report</summary>

```python
import json
from datetime import datetime, timezone

def build_evaluation_report(
    model_id,
    dataset_version,
    primary_metric,
    slice_metrics,
    robustness_metrics,
    config,
):
    """Create a machine-readable record of results and evaluation conditions."""
    return {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "model_id": model_id,
        "dataset_version": dataset_version,
        "metrics": {
            "primary": primary_metric,
            "slices": slice_metrics,
            "robustness": robustness_metrics,
        },
        "evaluation_config": config,
    }


report = build_evaluation_report(
    model_id="intent-classifier-v7",
    dataset_version="support-intents-2026-07",
    primary_metric={"name": "macro_f1", "value": 0.873},
    slice_metrics={
        "short_messages": {"macro_f1": 0.901, "n": 850},
        "long_messages": {"macro_f1": 0.811, "n": 210},
        "urgent_intent": {"recall": 0.957, "n": 140},
    },
    robustness_metrics={
        "typo_set_accuracy": 0.842,
        "ood_auroc": 0.913,
        "ece": 0.041,
    },
    config={
        "decision_threshold": 0.62,
        "tokenizer": "example-tokenizer-v3",
        "split_policy": "grouped by conversation_id and time",
        "random_seed": 42,
    },
)

with open("evaluation_report.json", "w", encoding="utf-8") as file:
    json.dump(report, file, ensure_ascii=False, indent=2)

print(json.dumps(report, ensure_ascii=False, indent=2))
```

</details>

Before releasing or comparing a model, the final report should answer these questions:

| Question | Evidence to include |
|---|---|
| what exact task is being measured? | input-output definition and label policy |
| does the data represent deployment? | source, time, domain, language, and slice distribution |
| what is the main success criterion? | pre-specified primary metric and operating point |
| what important failures can the average hide? | per-class, per-domain, worst-group, and robustness results |
| are differences reliable? | confidence intervals, paired tests, or repeated runs |
| can confidence be used operationally? | calibration and risk-coverage analysis |
| why does the model fail? | error taxonomy with representative examples |
| can someone reproduce the result? | versioned data, code, model, metric, prompt, and configuration |

The central lesson of NLP evaluation is that a metric is evidence about a specific behavior under a specific protocol. Strong evaluation combines automatic metrics, targeted behavioral tests, human judgment where necessary, robustness and fairness checks, uncertainty, and structured error analysis. The result is not merely a score; it is a defensible account of where the system works, where it fails, and what evidence supports deployment or further research.
